In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

import random
import os



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =========================
# LOAD N-BACK
# =========================
data_nb = torch.load(
    "/content/drive/MyDrive/eeg_preprocessed_nback.pt",
    weights_only=False
)

X_time_t = data_nb["X_time"]
X_bp_t   = data_nb["X_bp"]
X_cov_t  = data_nb["X_cov"]
X_plv_t  = data_nb["X_plv"]
y_t      = data_nb["y"]
subjects = data_nb["subjects"]

print("Loaded N-back")


# =========================
# LOAD HTC
# =========================
data_htc = torch.load(
    "/content/drive/MyDrive/eeg_preprocessed_htc.pt",
    weights_only=False
)

X_time_t_htc = data_htc["X_time"]
X_bp_t_htc   = data_htc["X_bp"]
X_cov_t_htc  = data_htc["X_cov"]
X_plv_t_htc  = data_htc["X_plv"]
y_t_htc      = data_htc["y"]
subjects_htc = data_htc["subjects"]

print("Loaded HTC")

Loaded N-back
Loaded HTC


In [ ]:
print("\n--- N-BACK ---")
print(X_time_t.shape, X_bp_t.shape, X_cov_t.shape, X_plv_t.shape, y_t.shape)

print("\n--- HTC ---")
print(X_time_t_htc.shape, X_bp_t_htc.shape, X_cov_t_htc.shape, X_plv_t_htc.shape, y_t_htc.shape)


--- N-BACK ---
torch.Size([12159, 14, 512]) torch.Size([12159, 42]) torch.Size([15007, 105]) torch.Size([15007, 210]) torch.Size([12159])

--- HTC ---
torch.Size([5056, 14, 512]) torch.Size([5056, 42]) torch.Size([5056, 105]) torch.Size([5056, 210]) torch.Size([5056])


In [ ]:
def extract_embeddings(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            db      = db.to(device)

            # SAFE: supports both versions of model
            try:
                e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi, db)
            except TypeError:
                e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)

In [ ]:
import numpy as np

# ----------------------------
# REMOVE 4 LOWEST HTC SUBJECTS
# ----------------------------
htc_perf = {
    "subject_01": 0.7677902579307556,
    "subject_02": 0.8880308866500854,
    "subject_03": 0.7392995953559875,
    "subject_06": 0.7952755689620972,
    "subject_08": 0.6611570119857788,
    "subject_12": 0.8793774247169495,
    "subject_16": 0.7346153855323792,
    "subject_17": 0.73046875,
    "subject_18": 0.8046875,
    "subject_19": 0.957198441028595,
    "subject_20": 0.6640926599502563,
    "subject_21": 0.8560311198234558,
    "subject_22": 0.8615384697914124,
    "subject_23": 0.8759689927101135,
    "subject_24": 0.9372549653053284,
    "subject_25": 0.6100385785102844,
    "subject_26": 0.8859315514564514,
}

K = 4
sorted_subjs = sorted(htc_perf.items(), key=lambda x: x[1])
remove_subjs = [s for s, _ in sorted_subjs[:K]]

print("Removing HTC subjects:", remove_subjs)

keep_mask = np.array([s not in remove_subjs for s in subjects_htc])

X_time_t_htc = X_time_t_htc[keep_mask]
X_bp_t_htc   = X_bp_t_htc[keep_mask]
X_cov_t_htc  = X_cov_t_htc[keep_mask]
X_plv_t_htc  = X_plv_t_htc[keep_mask]
y_t_htc      = y_t_htc[keep_mask]
subjects_htc = subjects_htc[keep_mask]

all_subjects_htc = np.unique(subjects_htc)
print("Remaining HTC subjects:", len(all_subjects_htc), all_subjects_htc)

Removing HTC subjects: ['subject_25', 'subject_08', 'subject_20', 'subject_17']
Remaining HTC subjects: 13 ['subject_01' 'subject_02' 'subject_03' 'subject_06' 'subject_12'
 'subject_16' 'subject_18' 'subject_19' 'subject_21' 'subject_22'
 'subject_23' 'subject_24' 'subject_26']


In [ ]:
all_subjects = np.unique(subjects)
print("\nSubjects list:", all_subjects)


Subjects list: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06'
 'subject_07' 'subject_08' 'subject_09' 'subject_10' 'subject_12'
 'subject_13' 'subject_14' 'subject_15']


In [ ]:
HTC_ID = 0
NBACK_ID = 1

subjects_nb_prefixed  = np.array([f"nb_{s}" for s in subjects])
subjects_htc_prefixed = np.array([f"htc_{s}" for s in subjects_htc])

print("Example N-back subjects:", subjects_nb_prefixed[:5])
print("Example HTC subjects:", subjects_htc_prefixed[:5])

Example N-back subjects: ['nb_subject_01' 'nb_subject_01' 'nb_subject_01' 'nb_subject_01'
 'nb_subject_01']
Example HTC subjects: ['htc_subject_01' 'htc_subject_01' 'htc_subject_01' 'htc_subject_01'
 'htc_subject_01']


In [ ]:
import torch
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs, dataset_id):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.X_roi  = X_roi
        self.y      = y

        self.dataset_id = int(dataset_id)

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx],
            torch.tensor(self.dataset_id, dtype=torch.long)
        )

In [ ]:
import torch.nn.functional as F

def supcon_within_dataset_loss(proj, y, subj, dataset, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.shape[0]

    sim = (proj @ proj.T) / temperature
    exp_sim = torch.exp(sim)

    y = y.unsqueeze(0)
    subj = subj.unsqueeze(0)
    dataset = dataset.unsqueeze(0)

    same_label   = (y == y.T)
    diff_subj    = (subj != subj.T)
    same_dataset = (dataset == dataset.T)

    positives = same_label & diff_subj & same_dataset

    eye = torch.eye(B, dtype=torch.bool, device=proj.device)
    positives = positives & ~eye
    valid = ~eye

    pos_exp = exp_sim * positives
    all_exp = exp_sim * valid

    pos_sum = pos_exp.sum(dim=1)
    denom   = all_exp.sum(dim=1) + 1e-8

    has_pos = positives.sum(dim=1) > 0
    if has_pos.sum() == 0:
        return torch.tensor(0.0, device=proj.device)

    log_prob = torch.log((pos_sum[has_pos] + 1e-8) / denom[has_pos])
    loss = -log_prob.mean()
    return loss

In [ ]:
from torch.utils.data import Sampler
import random
from collections import defaultdict

class JointBalancedBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, subjects_per_dataset=6):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_dataset = subjects_per_dataset

        self.group_to_indices = defaultdict(list)

        for idx in range(len(dataset)):
            _, _, _, _, _, _, subj, db = dataset[idx]
            key = (int(db.item()), int(subj.item()))
            self.group_to_indices[key].append(idx)

        self.nb_groups  = [g for g in self.group_to_indices if g[0] == NBACK_ID]
        self.htc_groups = [g for g in self.group_to_indices if g[0] == HTC_ID]

        assert len(self.nb_groups) > 0, "No N-back groups found"
        assert len(self.htc_groups) > 0, "No HTC groups found"

        self.nb_total  = sum(len(self.group_to_indices[g]) for g in self.nb_groups)
        self.htc_total = sum(len(self.group_to_indices[g]) for g in self.htc_groups)

        self.max_samples = max(self.nb_total, self.htc_total)
        self.num_batches = self.max_samples // batch_size

        self.samples_per_subject = batch_size // (2 * subjects_per_dataset)

    def __iter__(self):
        group_lists = {}
        for g in self.group_to_indices:
            lst = self.group_to_indices[g][:]
            random.shuffle(lst)
            group_lists[g] = lst

        group_ptrs = {g: 0 for g in group_lists}

        for _ in range(self.num_batches):
            chosen_nb  = random.sample(self.nb_groups,  self.subjects_per_dataset)
            chosen_htc = random.sample(self.htc_groups, self.subjects_per_dataset)

            batch = []

            for g in chosen_nb + chosen_htc:
                for _ in range(self.samples_per_subject):
                    if group_ptrs[g] >= len(group_lists[g]):
                        random.shuffle(group_lists[g])
                        group_ptrs[g] = 0

                    batch.append(group_lists[g][group_ptrs[g]])
                    group_ptrs[g] += 1

            yield batch

    def __len__(self):
        return self.num_batches

In [ ]:
def mixed_dataset_metric_loss(
    e, proj, yb, sb, db,
    lambda_con=0.5,
    proto_lambda=0.10,
    align_lambda=0.2,
    temperature=0.1
):
    # ----------------------------
    # 🔥 CRITICAL: normalize embeddings
    # ----------------------------
    e = F.normalize(e, dim=1)
    proj = F.normalize(proj, dim=1)

    loss_metric = torch.tensor(0.0, device=e.device)
    loss_proto  = torch.tensor(0.0, device=e.device)

    mask_nb  = (db == NBACK_ID)
    mask_htc = (db == HTC_ID)

    # ----------------------------
    # PROTOTYPE + METRIC (UNCHANGED)
    # ----------------------------
    if mask_nb.any():
        loss_metric = loss_metric + prototype_ce_loss(
            e[mask_nb], yb[mask_nb], temperature=temperature
        )
        loss_proto = loss_proto + prototype_loss(
            e[mask_nb], yb[mask_nb]
        )

    if mask_htc.any():
        loss_metric = loss_metric + prototype_ce_loss(
            e[mask_htc], yb[mask_htc], temperature=temperature
        )
        loss_proto = loss_proto + prototype_loss(
            e[mask_htc], yb[mask_htc]
        )

    # ----------------------------
    # SUPCON (UNCHANGED)
    # ----------------------------
    loss_con = supcon_within_dataset_loss(
        proj, yb, sb, db, temperature=temperature
    )

    # ----------------------------
    # 🔥 NEW: PROTOTYPE DIRECTION ALIGNMENT
    # ----------------------------
    loss_align = torch.tensor(0.0, device=e.device)

    if mask_nb.any() and mask_htc.any():

        # ----- N-BACK -----
        e_nb = e[mask_nb]
        y_nb = yb[mask_nb]

        if len(torch.unique(y_nb)) >= 2:
            c_min = e_nb[y_nb == y_nb.min()].mean(dim=0)
            c_max = e_nb[y_nb == y_nb.max()].mean(dim=0)
            nb_vec = F.normalize(c_max - c_min, dim=0)
        else:
            nb_vec = None

        # ----- HTC -----
        e_htc = e[mask_htc]
        y_htc = yb[mask_htc]

        if len(torch.unique(y_htc)) >= 2:
            c_low = e_htc[y_htc == y_htc.min()].mean(dim=0)
            c_high = e_htc[y_htc == y_htc.max()].mean(dim=0)
            htc_vec = F.normalize(c_high - c_low, dim=0)
        else:
            htc_vec = None

        if nb_vec is not None and htc_vec is not None:
            loss_align = 1 - torch.dot(nb_vec, htc_vec)

    # ----------------------------
    # FINAL LOSS
    # ----------------------------
    total = (
        loss_metric
        + lambda_con * loss_con
        + proto_lambda * loss_proto
        + align_lambda * loss_align
    )

    return total, loss_metric, loss_con, loss_proto

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionEncoder(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=3):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim   # 42 + 3 = 45

        # ------------------
        # TIME BRANCH
        # ------------------
        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        # ------------------
        # BANDPOWER + ROI BRANCH
        # ------------------
        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),   # was 42 → now 45
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        # ------------------
        # COVARIANCE BRANCH
        # ------------------
        self.cov_branch = nn.Sequential(
            nn.Linear(105, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # PLV BRANCH
        # ------------------
        self.plv_branch = nn.Sequential(
            nn.Linear(210, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # FUSION
        # ------------------
        self.embed = nn.Sequential(
            nn.Linear(32 + 32 + 32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):

        # ------------------
        # Time branch
        # ------------------
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)  # (B,32)

        # ------------------
        # Bandpower + ROI
        # ------------------
        bp_in = torch.cat([x_bp, x_roi], dim=1)  # (B,45)
        z_bp = self.bp_branch(bp_in)

        # ------------------
        # Covariance
        # ------------------
        z_cov = self.cov_branch(x_cov)

        # ------------------
        # PLV
        # ------------------
        z_plv = self.plv_branch(x_plv)

        # ------------------
        # Fusion
        # ------------------
        z = torch.cat([z_time, z_bp, z_cov, z_plv], dim=1)
        e = self.embed(z)

        return e

In [ ]:
import torch.nn as nn

class ContrastiveModel(nn.Module):
    def __init__(self,
                 emb_dim=64,
                 proj_dim=64,
                 bp_dim=42,
                 roi_dim=3):

        super().__init__()

        # Main encoder (learns embedding geometry)
        self.encoder = FusionEncoder(
            emb_dim=emb_dim,
            bp_dim=bp_dim,
            roi_dim=roi_dim
        )

        # Projection head for SupCon loss
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):
        # Embedding
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)

        # Projection (used only for contrastive loss)
        p = self.proj(e)

        return e, p

In [ ]:
def prototype_loss(emb, y):
    """
    emb: (B, D) embedding vectors (from encoder)
    y:   (B,) class labels
    """

    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]              # (Nc, D)
        proto = class_emb.mean(dim=0)      # (D,)

        loss += ((class_emb - proto)**2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)

In [ ]:
import torch.nn as nn

def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    """
    Put BN layers in *specified submodules* into train mode (update running stats).
    Keep all other BN in eval mode.
    Always keep Dropout in eval mode.
    """
    # Default everything to eval
    model.eval()

    # Dropout always eval
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    # Enable BN train only in allowed named submodules
    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()

In [ ]:
import torch.nn.functional as F

def prototype_ce_loss(e, y, temperature=0.1):
    """
    e: (B, D) embeddings
    y: (B,) labels
    """

    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)  # (C, D)

    # normalize for cosine
    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T  # (B, C)
    logits = logits / temperature

    # map labels to [0, C-1]
    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y],
                            device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))
overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLIT
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb  = np.where(train_mask_nb)[0]
    test_idx_nb   = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc  = np.where(train_mask_htc)[0]
    test_idx_htc   = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0, 2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0, 2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    sampler = JointBalancedBatchSampler(
        train_ds_joint,
        batch_size=BATCH,
        subjects_per_dataset=6
    )

    train_loader         = DataLoader(train_ds_joint, batch_sampler=sampler)
    train_eval_loader_nb = DataLoader(train_ds_nb, batch_size=256, shuffle=False)
    train_eval_loader_htc = DataLoader(train_ds_htc, batch_size=256, shuffle=False)
    test_loader_nb       = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc      = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_score = -1.0
    best_state = None
    best_acc_nb = -1.0
    best_acc_htc = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, loss_metric, loss_con, loss_proto = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL: NB
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(model, train_eval_loader_nb, device)
            e_test_nb,  y_test_nb_full  = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0) for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )

            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            # =========================
            # ZERO-SHOT EVAL: HTC
            # =========================
            e_train_htc, y_train_htc_full = extract_embeddings(model, train_eval_loader_htc, device)
            e_test_htc,  y_test_htc_full  = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0) for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )

            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_acc_nb = acc_nb
            best_acc_htc = acc_htc

        print(f"Epoch {ep:02d} | NB zero-shot {acc_nb:.4f} | HTC zero-shot {acc_htc:.4f} | avg {score:.4f}")

    # =====================================================
    # LOAD BEST BASE MODEL
    # =====================================================
    model.load_state_dict(best_state)
    best_joint_state = copy.deepcopy(best_state)

    # =====================================================
    # SHIFT COMPUTATION ON BEST BASE MODEL
    # =====================================================
    model.eval()
    with torch.no_grad():
        e_train_nb_base, y_train_nb_base = extract_embeddings(model, train_eval_loader_nb, device)
        e_test_nb_base,  y_test_nb_base  = extract_embeddings(model, test_loader_nb, device)

        e_train_htc_base, y_train_htc_base = extract_embeddings(model, train_eval_loader_htc, device)
        e_test_htc_base,  y_test_htc_base  = extract_embeddings(model, test_loader_htc, device)

    shift_nb = torch.norm(e_train_nb_base.mean(0) - e_test_nb_base.mean(0)).item()
    shift_htc = torch.norm(e_train_htc_base.mean(0) - e_test_htc_base.mean(0)).item()

    print(f"Best-model shift | NB: {shift_nb:.4f} | HTC: {shift_htc:.4f}")

    # =====================================================
    # FEW-SHOT: N-BACK (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    # start NB adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_nb = e_test_nb_base.clone()
    all_y_nb = y_test_nb_base.clone()

    support_idx_nb = []
    for c in torch.unique(all_y_nb):
        class_idx = (all_y_nb == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_nb.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_nb = torch.cat(support_idx_nb)

    mask_nb = torch.zeros(len(all_y_nb), dtype=torch.bool)
    mask_nb[support_idx_nb] = True

    support_subset_nb = torch.utils.data.Subset(test_ds_nb, support_idx_nb.tolist())
    support_loader_nb = DataLoader(support_subset_nb, batch_size=256, shuffle=False)

    use_bn_nb = shift_nb > SHIFT_THRESH
    print(f"N-back BN USED: {use_bn_nb}")

    if use_bn_nb:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_nb:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_nb, all_y_nb = extract_embeddings(model, test_loader_nb, device)

    e_support_nb = all_e_nb[mask_nb].to(device)
    y_support_nb = all_y_nb[mask_nb].to(device)
    e_query_nb   = all_e_nb[~mask_nb].to(device)
    y_query_nb   = all_y_nb[~mask_nb].to(device)

    e_support_nb = F.normalize(e_support_nb, dim=1)
    e_query_nb   = F.normalize(e_query_nb, dim=1)

    var_nb = e_support_nb.var(dim=0)
    weights_nb = 1.0 / (var_nb + 1e-6)
    weights_nb = weights_nb / weights_nb.mean()
    e_support_nb = e_support_nb * weights_nb
    e_query_nb   = e_query_nb * weights_nb

    classes_nb = torch.unique(y_support_nb)
    protos_nb = torch.stack([
        e_support_nb[y_support_nb == c].mean(dim=0) for c in classes_nb
    ])
    protos_nb = F.normalize(protos_nb, dim=1)

    residuals_nb = torch.cat([
        e_support_nb[y_support_nb == c] - protos_nb[i]
        for i, c in enumerate(classes_nb)
    ], dim=0)

    D_nb = residuals_nb.shape[1]
    cov_nb = (residuals_nb.t() @ residuals_nb) / max(residuals_nb.shape[0] - 1, 1)

    avg_var_nb = torch.trace(cov_nb) / D_nb
    cov_shrink_nb = (1 - SHRINK) * cov_nb + SHRINK * (avg_var_nb * torch.eye(D_nb, device=device))
    cov_inv_nb = torch.linalg.pinv(cov_shrink_nb)

    diffs_nb = e_query_nb.unsqueeze(1) - protos_nb.unsqueeze(0)
    dists_nb = torch.einsum("nkd,dd,nkd->nk", diffs_nb, cov_inv_nb, diffs_nb)

    pred_nb_fs = dists_nb.argmin(dim=1)
    fewshot_acc_nb = (pred_nb_fs == y_query_nb).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_nb)

    # =====================================================
    # FEW-SHOT: HTC (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(2234 + fold_i)
    np.random.seed(2234 + fold_i)

    # start HTC adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_htc = e_test_htc_base.clone()
    all_y_htc = y_test_htc_base.clone()

    support_idx_htc = []
    for c in torch.unique(all_y_htc):
        class_idx = (all_y_htc == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_htc.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_htc = torch.cat(support_idx_htc)

    mask_htc = torch.zeros(len(all_y_htc), dtype=torch.bool)
    mask_htc[support_idx_htc] = True

    support_subset_htc = torch.utils.data.Subset(test_ds_htc, support_idx_htc.tolist())
    support_loader_htc = DataLoader(support_subset_htc, batch_size=256, shuffle=False)

    use_bn_htc = shift_htc > SHIFT_THRESH
    print(f"HTC BN USED: {use_bn_htc}")

    if use_bn_htc:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_htc:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_htc, all_y_htc = extract_embeddings(model, test_loader_htc, device)

    e_support_htc = all_e_htc[mask_htc].to(device)
    y_support_htc = all_y_htc[mask_htc].to(device)
    e_query_htc   = all_e_htc[~mask_htc].to(device)
    y_query_htc   = all_y_htc[~mask_htc].to(device)

    e_support_htc = F.normalize(e_support_htc, dim=1)
    e_query_htc   = F.normalize(e_query_htc, dim=1)

    var_htc = e_support_htc.var(dim=0)
    weights_htc = 1.0 / (var_htc + 1e-6)
    weights_htc = weights_htc / weights_htc.mean()
    e_support_htc = e_support_htc * weights_htc
    e_query_htc   = e_query_htc * weights_htc

    classes_htc = torch.unique(y_support_htc)
    protos_htc = torch.stack([
        e_support_htc[y_support_htc == c].mean(dim=0) for c in classes_htc
    ])
    protos_htc = F.normalize(protos_htc, dim=1)

    residuals_htc = torch.cat([
        e_support_htc[y_support_htc == c] - protos_htc[i]
        for i, c in enumerate(classes_htc)
    ], dim=0)

    D_htc = residuals_htc.shape[1]
    cov_htc = (residuals_htc.t() @ residuals_htc) / max(residuals_htc.shape[0] - 1, 1)

    avg_var_htc = torch.trace(cov_htc) / D_htc
    cov_shrink_htc = (1 - SHRINK) * cov_htc + SHRINK * (avg_var_htc * torch.eye(D_htc, device=device))
    cov_inv_htc = torch.linalg.pinv(cov_shrink_htc)

    diffs_htc = e_query_htc.unsqueeze(1) - protos_htc.unsqueeze(0)
    dists_htc = torch.einsum("nkd,dd,nkd->nk", diffs_htc, cov_inv_htc, diffs_htc)

    pred_htc_fs = dists_htc.argmin(dim=1)
    fewshot_acc_htc = (pred_htc_fs == y_query_htc).float().mean().item()

    print("HTC gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_htc)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": best_acc_nb,
        "best_zero_shot_htc": best_acc_htc,
        "shift_nb": shift_nb,
        "shift_htc": shift_htc,
        "bn_used_nb": bool(use_bn_nb),
        "bn_used_htc": bool(use_bn_htc),
        "fewshot_nb": fewshot_acc_nb,
        "fewshot_htc": fewshot_acc_htc
    })

print("\nJOINT METRIC TRAIN + GATED ADABN + MAHALANOBIS LOSO SUMMARY")
for r in overall_results_joint:
    print(
        f"Fold {r['fold']:02d} | "
        f"NB test {r['test_subj_nb']} | HTC test {r['test_subj_htc']} | "
        f"NB zero-shot {r['best_zero_shot_nb']:.4f} | "
        f"HTC zero-shot {r['best_zero_shot_htc']:.4f} | "
        f"NB shift {r['shift_nb']:.4f} | HTC shift {r['shift_htc']:.4f} | "
        f"NB BN {r['bn_used_nb']} | HTC BN {r['bn_used_htc']} | "
        f"NB few-shot {r['fewshot_nb']:.4f} | "
        f"HTC few-shot {r['fewshot_htc']:.4f}"
    )

nb_fs_accs  = [r["fewshot_nb"] for r in overall_results_joint]
htc_fs_accs = [r["fewshot_htc"] for r in overall_results_joint]

print("\nNB few-shot mean/std:", float(np.mean(nb_fs_accs)), float(np.std(nb_fs_accs)))
print("HTC few-shot mean/std:", float(np.mean(htc_fs_accs)), float(np.std(htc_fs_accs)))


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Epoch 01 | NB zero-shot 0.5983 | HTC zero-shot 0.6319 | avg 0.6151
Epoch 02 | NB zero-shot 0.6141 | HTC zero-shot 0.6580 | avg 0.6360
Epoch 03 | NB zero-shot 0.6225 | HTC zero-shot 0.6384 | avg 0.6305
Epoch 04 | NB zero-shot 0.6677 | HTC zero-shot 0.6221 | avg 0.6449
Epoch 05 | NB zero-shot 0.6898 | HTC zero-shot 0.6124 | avg 0.6511
Epoch 06 | NB zero-shot 0.7213 | HTC zero-shot 0.6156 | avg 0.6685
Epoch 07 | NB zero-shot 0.7129 | HTC zero-shot 0.6059 | avg 0.6594
Epoch 08 | NB zero-shot 0.7192 | HTC zero-shot 0.5928 | avg 0.6560
Epoch 09 | NB zero-shot 0.7014 | HTC zero-shot 0.6026 | avg 0.6520
Epoch 10 | NB zero-shot 0.7581 | HTC zero-shot 0.5896 | avg 0.6739
Best-model shift | NB: 0.9555 | HTC: 1.9964
N-back BN USED: True
N-back gated AdaBN + Mahalanobis few-shot acc: 0.8092032074928284
HTC BN USED: True
HTC gated AdaBN + Mahalanobis few-shot acc: 0.7902621626853943

JOINT FOLD 2/13
N-back test subject:

In [ ]:
'''
🔵 NEW ALIGNMENT — WHAT IT DOES

Now it looks at distances.

For N-back:

Distances:

d01 = distance(0,1)
d12 = distance(1,2)
d02 = distance(0,2)
For HTC:
dLH = distance(low, high)
Then your code compares:
d01  vs  dLH

(just the first one because of size mismatch)

🚨 THIS IS THE KEY DIFFERENCE
OLD compares:
a VECTOR (direction)
NEW compares:
a DISTANCE (spacing)
🧠 NOW THE IMPORTANT PART (WHY YOU’RE CONFUSED)

You’re thinking:

“But new is comparing the wrong things”

YES. It is.

But here’s what it STILL does:

🔥 WHAT NEW IS FORCING

It forces:

distance(0,1) ≈ distance(low, high)
🧠 WHAT THAT MEANS FOR THE MODEL

The model is now pressured to:

make class gaps consistent

So instead of this:

0 —— 1 ——————— 2   (bad spacing)

It becomes more like:

0 —— 1 —— 2   (even spacing)
🔴 WHY OLD DOESN’T DO THIS

OLD only cares about:

(2 - 0)

So this is ALSO valid for OLD:

0 ——————— 1 — 2

Same direction → OLD is happy

🔥 WHY NEW HELPS PERFORMANCE

Because your classifier uses:

distance to prototypes

NOT direction.

🧠 SIMPLE FINAL DIFFERENCE
OLD:
checks:
→ direction of endpoints
NEW:
pushes:
→ spacing between classes
⚡ SUPER SIMPLE ANALOGY (last one, I promise)

You have 3 points:

A   B   C
OLD says:
just make A → C point right
NEW says:
make distance(A,B) reasonable
'''

In [ ]:
import torch
import torch.nn.functional as F

def mixed_dataset_metric_loss(
    e, proj, yb, sb, db,
    lambda_con=0.5,
    proto_lambda=0.10,
    temperature=0.1
):
    # ----------------------------
    # normalize embeddings
    # ----------------------------
    e = F.normalize(e, dim=1)
    proj = F.normalize(proj, dim=1)

    loss_metric = torch.tensor(0.0, device=e.device)
    loss_proto  = torch.tensor(0.0, device=e.device)

    mask_nb  = (db == NBACK_ID)
    mask_htc = (db == HTC_ID)

    # ----------------------------
    # PROTOTYPE + METRIC
    # ----------------------------
    if mask_nb.any():
        loss_metric += prototype_ce_loss(
            e[mask_nb], yb[mask_nb], temperature=temperature
        )
        loss_proto += prototype_loss(
            e[mask_nb], yb[mask_nb]
        )

    if mask_htc.any():
        loss_metric += prototype_ce_loss(
            e[mask_htc], yb[mask_htc], temperature=temperature
        )
        loss_proto += prototype_loss(
            e[mask_htc], yb[mask_htc]
        )

    # ----------------------------
    # SUPCON
    # ----------------------------
    loss_con = supcon_within_dataset_loss(
        proj, yb, sb, db, temperature=temperature
    )

    # ----------------------------
    # 🔥 FIXED: DISTANCE CONSISTENCY
    # ----------------------------
    loss_dist = torch.tensor(0.0, device=e.device)

    if mask_nb.any() and mask_htc.any():

        def get_dist_vector(e_sub, y_sub):
            classes = torch.unique(y_sub)

            if len(classes) < 2:
                return None

            protos = torch.stack([
                e_sub[y_sub == c].mean(dim=0)
                for c in classes
            ])

            protos = F.normalize(protos, dim=1)

            # pairwise distances
            dists = torch.cdist(protos, protos, p=2)

            # 🔥 take upper triangle only (ignore size mismatch)
            idx = torch.triu_indices(dists.size(0), dists.size(1), offset=1)
            vec = dists[idx[0], idx[1]]

            return vec

        d_nb  = get_dist_vector(e[mask_nb], yb[mask_nb])
        d_htc = get_dist_vector(e[mask_htc], yb[mask_htc])

        if d_nb is not None and d_htc is not None:

            # normalize
            d_nb  = d_nb  / (d_nb.mean()  + 1e-6)
            d_htc = d_htc / (d_htc.mean() + 1e-6)

            # 🔥 match only overlapping length
            L = min(len(d_nb), len(d_htc))

            if L > 0:
                loss_dist = F.mse_loss(d_nb[:L], d_htc[:L])

    # ----------------------------
    # FINAL LOSS
    # ----------------------------
    total = (
        loss_metric
        + lambda_con * loss_con
        + proto_lambda * loss_proto
        + 0.1 * loss_dist
    )

    return total, loss_metric, loss_con, loss_proto

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))
overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLIT
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb  = np.where(train_mask_nb)[0]
    test_idx_nb   = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc  = np.where(train_mask_htc)[0]
    test_idx_htc   = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0, 2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0, 2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    sampler = JointBalancedBatchSampler(
        train_ds_joint,
        batch_size=BATCH,
        subjects_per_dataset=6
    )

    train_loader         = DataLoader(train_ds_joint, batch_sampler=sampler)
    train_eval_loader_nb = DataLoader(train_ds_nb, batch_size=256, shuffle=False)
    train_eval_loader_htc = DataLoader(train_ds_htc, batch_size=256, shuffle=False)
    test_loader_nb       = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc      = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_score = -1.0
    best_state = None
    best_acc_nb = -1.0
    best_acc_htc = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, loss_metric, loss_con, loss_proto = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL: NB
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(model, train_eval_loader_nb, device)
            e_test_nb,  y_test_nb_full  = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0) for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )

            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            # =========================
            # ZERO-SHOT EVAL: HTC
            # =========================
            e_train_htc, y_train_htc_full = extract_embeddings(model, train_eval_loader_htc, device)
            e_test_htc,  y_test_htc_full  = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0) for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )

            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_acc_nb = acc_nb
            best_acc_htc = acc_htc

        print(f"Epoch {ep:02d} | NB zero-shot {acc_nb:.4f} | HTC zero-shot {acc_htc:.4f} | avg {score:.4f}")

    # =====================================================
    # LOAD BEST BASE MODEL
    # =====================================================
    model.load_state_dict(best_state)
    best_joint_state = copy.deepcopy(best_state)

    # =====================================================
    # SHIFT COMPUTATION ON BEST BASE MODEL
    # =====================================================
    model.eval()
    with torch.no_grad():
        e_train_nb_base, y_train_nb_base = extract_embeddings(model, train_eval_loader_nb, device)
        e_test_nb_base,  y_test_nb_base  = extract_embeddings(model, test_loader_nb, device)

        e_train_htc_base, y_train_htc_base = extract_embeddings(model, train_eval_loader_htc, device)
        e_test_htc_base,  y_test_htc_base  = extract_embeddings(model, test_loader_htc, device)

    shift_nb = torch.norm(e_train_nb_base.mean(0) - e_test_nb_base.mean(0)).item()
    shift_htc = torch.norm(e_train_htc_base.mean(0) - e_test_htc_base.mean(0)).item()

    print(f"Best-model shift | NB: {shift_nb:.4f} | HTC: {shift_htc:.4f}")

    # =====================================================
    # FEW-SHOT: N-BACK (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    # start NB adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_nb = e_test_nb_base.clone()
    all_y_nb = y_test_nb_base.clone()

    support_idx_nb = []
    for c in torch.unique(all_y_nb):
        class_idx = (all_y_nb == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_nb.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_nb = torch.cat(support_idx_nb)

    mask_nb = torch.zeros(len(all_y_nb), dtype=torch.bool)
    mask_nb[support_idx_nb] = True

    support_subset_nb = torch.utils.data.Subset(test_ds_nb, support_idx_nb.tolist())
    support_loader_nb = DataLoader(support_subset_nb, batch_size=256, shuffle=False)

    use_bn_nb = shift_nb > SHIFT_THRESH
    print(f"N-back BN USED: {use_bn_nb}")

    if use_bn_nb:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_nb:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_nb, all_y_nb = extract_embeddings(model, test_loader_nb, device)

    e_support_nb = all_e_nb[mask_nb].to(device)
    y_support_nb = all_y_nb[mask_nb].to(device)
    e_query_nb   = all_e_nb[~mask_nb].to(device)
    y_query_nb   = all_y_nb[~mask_nb].to(device)

    e_support_nb = F.normalize(e_support_nb, dim=1)
    e_query_nb   = F.normalize(e_query_nb, dim=1)

    var_nb = e_support_nb.var(dim=0)
    weights_nb = 1.0 / (var_nb + 1e-6)
    weights_nb = weights_nb / weights_nb.mean()
    e_support_nb = e_support_nb * weights_nb
    e_query_nb   = e_query_nb * weights_nb

    classes_nb = torch.unique(y_support_nb)
    protos_nb = torch.stack([
        e_support_nb[y_support_nb == c].mean(dim=0) for c in classes_nb
    ])
    protos_nb = F.normalize(protos_nb, dim=1)

    residuals_nb = torch.cat([
        e_support_nb[y_support_nb == c] - protos_nb[i]
        for i, c in enumerate(classes_nb)
    ], dim=0)

    D_nb = residuals_nb.shape[1]
    cov_nb = (residuals_nb.t() @ residuals_nb) / max(residuals_nb.shape[0] - 1, 1)

    avg_var_nb = torch.trace(cov_nb) / D_nb
    cov_shrink_nb = (1 - SHRINK) * cov_nb + SHRINK * (avg_var_nb * torch.eye(D_nb, device=device))
    cov_inv_nb = torch.linalg.pinv(cov_shrink_nb)

    diffs_nb = e_query_nb.unsqueeze(1) - protos_nb.unsqueeze(0)
    dists_nb = torch.einsum("nkd,dd,nkd->nk", diffs_nb, cov_inv_nb, diffs_nb)

    pred_nb_fs = dists_nb.argmin(dim=1)
    fewshot_acc_nb = (pred_nb_fs == y_query_nb).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_nb)

    # =====================================================
    # FEW-SHOT: HTC (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(2234 + fold_i)
    np.random.seed(2234 + fold_i)

    # start HTC adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_htc = e_test_htc_base.clone()
    all_y_htc = y_test_htc_base.clone()

    support_idx_htc = []
    for c in torch.unique(all_y_htc):
        class_idx = (all_y_htc == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_htc.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_htc = torch.cat(support_idx_htc)

    mask_htc = torch.zeros(len(all_y_htc), dtype=torch.bool)
    mask_htc[support_idx_htc] = True

    support_subset_htc = torch.utils.data.Subset(test_ds_htc, support_idx_htc.tolist())
    support_loader_htc = DataLoader(support_subset_htc, batch_size=256, shuffle=False)

    use_bn_htc = shift_htc > SHIFT_THRESH
    print(f"HTC BN USED: {use_bn_htc}")

    if use_bn_htc:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_htc:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_htc, all_y_htc = extract_embeddings(model, test_loader_htc, device)

    e_support_htc = all_e_htc[mask_htc].to(device)
    y_support_htc = all_y_htc[mask_htc].to(device)
    e_query_htc   = all_e_htc[~mask_htc].to(device)
    y_query_htc   = all_y_htc[~mask_htc].to(device)

    e_support_htc = F.normalize(e_support_htc, dim=1)
    e_query_htc   = F.normalize(e_query_htc, dim=1)

    var_htc = e_support_htc.var(dim=0)
    weights_htc = 1.0 / (var_htc + 1e-6)
    weights_htc = weights_htc / weights_htc.mean()
    e_support_htc = e_support_htc * weights_htc
    e_query_htc   = e_query_htc * weights_htc

    classes_htc = torch.unique(y_support_htc)
    protos_htc = torch.stack([
        e_support_htc[y_support_htc == c].mean(dim=0) for c in classes_htc
    ])
    protos_htc = F.normalize(protos_htc, dim=1)

    residuals_htc = torch.cat([
        e_support_htc[y_support_htc == c] - protos_htc[i]
        for i, c in enumerate(classes_htc)
    ], dim=0)

    D_htc = residuals_htc.shape[1]
    cov_htc = (residuals_htc.t() @ residuals_htc) / max(residuals_htc.shape[0] - 1, 1)

    avg_var_htc = torch.trace(cov_htc) / D_htc
    cov_shrink_htc = (1 - SHRINK) * cov_htc + SHRINK * (avg_var_htc * torch.eye(D_htc, device=device))
    cov_inv_htc = torch.linalg.pinv(cov_shrink_htc)

    diffs_htc = e_query_htc.unsqueeze(1) - protos_htc.unsqueeze(0)
    dists_htc = torch.einsum("nkd,dd,nkd->nk", diffs_htc, cov_inv_htc, diffs_htc)

    pred_htc_fs = dists_htc.argmin(dim=1)
    fewshot_acc_htc = (pred_htc_fs == y_query_htc).float().mean().item()

    print("HTC gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_htc)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": best_acc_nb,
        "best_zero_shot_htc": best_acc_htc,
        "shift_nb": shift_nb,
        "shift_htc": shift_htc,
        "bn_used_nb": bool(use_bn_nb),
        "bn_used_htc": bool(use_bn_htc),
        "fewshot_nb": fewshot_acc_nb,
        "fewshot_htc": fewshot_acc_htc
    })

print("\nJOINT METRIC TRAIN + GATED ADABN + MAHALANOBIS LOSO SUMMARY")
for r in overall_results_joint:
    print(
        f"Fold {r['fold']:02d} | "
        f"NB test {r['test_subj_nb']} | HTC test {r['test_subj_htc']} | "
        f"NB zero-shot {r['best_zero_shot_nb']:.4f} | "
        f"HTC zero-shot {r['best_zero_shot_htc']:.4f} | "
        f"NB shift {r['shift_nb']:.4f} | HTC shift {r['shift_htc']:.4f} | "
        f"NB BN {r['bn_used_nb']} | HTC BN {r['bn_used_htc']} | "
        f"NB few-shot {r['fewshot_nb']:.4f} | "
        f"HTC few-shot {r['fewshot_htc']:.4f}"
    )

nb_fs_accs  = [r["fewshot_nb"] for r in overall_results_joint]
htc_fs_accs = [r["fewshot_htc"] for r in overall_results_joint]

print("\nNB few-shot mean/std:", float(np.mean(nb_fs_accs)), float(np.std(nb_fs_accs)))
print("HTC few-shot mean/std:", float(np.mean(htc_fs_accs)), float(np.std(htc_fs_accs)))


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Epoch 01 | NB zero-shot 0.6004 | HTC zero-shot 0.5733 | avg 0.5869
Epoch 02 | NB zero-shot 0.6898 | HTC zero-shot 0.5928 | avg 0.6413
Epoch 03 | NB zero-shot 0.7182 | HTC zero-shot 0.5961 | avg 0.6571
Epoch 04 | NB zero-shot 0.7234 | HTC zero-shot 0.6221 | avg 0.6728
Epoch 05 | NB zero-shot 0.7256 | HTC zero-shot 0.5407 | avg 0.6331
Epoch 06 | NB zero-shot 0.7739 | HTC zero-shot 0.5700 | avg 0.6720
Epoch 07 | NB zero-shot 0.7655 | HTC zero-shot 0.5179 | avg 0.6417
Epoch 08 | NB zero-shot 0.8065 | HTC zero-shot 0.5928 | avg 0.6997
Epoch 09 | NB zero-shot 0.7729 | HTC zero-shot 0.5635 | avg 0.6682
Epoch 10 | NB zero-shot 0.7739 | HTC zero-shot 0.5733 | avg 0.6736
Best-model shift | NB: 0.9071 | HTC: 1.6273
N-back BN USED: True
N-back gated AdaBN + Mahalanobis few-shot acc: 0.8585858941078186
HTC BN USED: True
HTC gated AdaBN + Mahalanobis few-shot acc: 0.7715355753898621

JOINT FOLD 2/13
N-back test subject:

In [ ]:
'''
🔥 THE ONE CHANGE THAT MATTERS

You changed this:

# OLD
loss_con = supcon_within_dataset_loss(proj, yb, sb, db)

to:

# NEW
loss_con = supcon_within_dataset_loss(e, yb, sb, db)
🧠 WHAT THAT MEANS (NO BS)
BEFORE:
SupCon operates on:  proj (projection head output)
NOW:
SupCon operates on:  e (main embedding)
⚠️ WHY THIS IS A BIG DEAL

Because:

e = what your classifier uses
proj = auxiliary space used ONLY for contrastive loss
🔴 OLD BEHAVIOR
Flow:
encoder → e → projection head → proj → SupCon loss

So:

👉 SupCon shapes proj space, NOT embedding space

Consequence:
Encoder can “cheat”
Projection head absorbs contrastive constraints
Embedding e is mostly shaped by:
prototype loss
CE loss
🔵 NEW BEHAVIOR
Flow:
encoder → e → SupCon loss directly

👉 Now SupCon directly shapes the embedding space

🔥 WHAT THIS DOES IN PRACTICE
1. Stronger clustering

Now:

same class → pulled together in e
different class → pushed apart in e

Before:

that only happened in proj (not used later)
2. Removes “projection escape hatch”

Before:

👉 model could do:

e = messy
proj = clean

Now:

👉 model MUST do:

e = clean
3. Directly improves classifier

Your classifier uses:

e → prototypes → cosine / Mahalanobis

So now:

👉 SupCon directly improves what you actually use

🔥 INTERACTION WITH YOUR DISTANCE LOSS

Now this combo is happening:

SupCon (on e)  → pulls same-class together
Prototype loss → tight clusters
Distance loss  → consistent spacing
Before it was:
SupCon → proj (irrelevant later)
Prototype → e
Distance → e
Now it’s:
SupCon → e  ✅
Prototype → e
Distance → e

👉 Everything is shaping the SAME space




⚠️ TRADEOFF (IMPORTANT)

This change makes training:

more powerful BUT more fragile

Because:

SupCon gradients hit encoder directly
Can overpower prototype loss if too strong
🧠 SIMPLE SUMMARY
OLD:
SupCon trains a SIDE space (proj)
embedding stays partially unconstrained
NEW:
SupCon trains the ACTUAL embedding (e)
everything is aligned


'''

In [ ]:
import torch
import torch.nn.functional as F

def mixed_dataset_metric_loss(
    e, proj, yb, sb, db,
    lambda_con=0.5,
    proto_lambda=0.10,
    temperature=0.1
):
    # ----------------------------
    # normalize embeddings
    # ----------------------------
    e = F.normalize(e, dim=1)
    proj = F.normalize(proj, dim=1)

    loss_metric = torch.tensor(0.0, device=e.device)
    loss_proto  = torch.tensor(0.0, device=e.device)

    mask_nb  = (db == NBACK_ID)
    mask_htc = (db == HTC_ID)

    # ----------------------------
    # PROTOTYPE + METRIC
    # ----------------------------
    if mask_nb.any():
        loss_metric += prototype_ce_loss(
            e[mask_nb], yb[mask_nb], temperature=temperature
        )
        loss_proto += prototype_loss(
            e[mask_nb], yb[mask_nb]
        )

    if mask_htc.any():
        loss_metric += prototype_ce_loss(
            e[mask_htc], yb[mask_htc], temperature=temperature
        )
        loss_proto += prototype_loss(
            e[mask_htc], yb[mask_htc]
        )

    # ----------------------------
    # SUPCON
    # ----------------------------
    # loss_con = supcon_within_dataset_loss(
    #     proj, yb, sb, db, temperature=temperature
    # )

    loss_con = supcon_within_dataset_loss(
          e, yb, sb, db, temperature=temperature
    )



    # ----------------------------
    # 🔥 FIXED: DISTANCE CONSISTENCY
    # ----------------------------
    loss_dist = torch.tensor(0.0, device=e.device)

    if mask_nb.any() and mask_htc.any():

        def get_dist_vector(e_sub, y_sub):
            classes = torch.unique(y_sub)

            if len(classes) < 2:
                return None

            protos = torch.stack([
                e_sub[y_sub == c].mean(dim=0)
                for c in classes
            ])

            protos = F.normalize(protos, dim=1)

            # pairwise distances
            dists = torch.cdist(protos, protos, p=2)

            # 🔥 take upper triangle only (ignore size mismatch)
            idx = torch.triu_indices(dists.size(0), dists.size(1), offset=1)
            vec = dists[idx[0], idx[1]]

            return vec

        d_nb  = get_dist_vector(e[mask_nb], yb[mask_nb])
        d_htc = get_dist_vector(e[mask_htc], yb[mask_htc])

        if d_nb is not None and d_htc is not None:

            # normalize
            d_nb  = d_nb  / (d_nb.mean()  + 1e-6)
            d_htc = d_htc / (d_htc.mean() + 1e-6)

            # 🔥 match only overlapping length
            L = min(len(d_nb), len(d_htc))

            if L > 0:
                loss_dist = F.mse_loss(d_nb[:L], d_htc[:L])

    # ----------------------------
    # FINAL LOSS
    # ----------------------------
    total = (
        loss_metric
        + lambda_con * loss_con
        + proto_lambda * loss_proto
        + 0.1 * loss_dist
    )

    return total, loss_metric, loss_con, loss_proto

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))
overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLIT
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb  = np.where(train_mask_nb)[0]
    test_idx_nb   = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc  = np.where(train_mask_htc)[0]
    test_idx_htc   = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0, 2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0, 2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    sampler = JointBalancedBatchSampler(
        train_ds_joint,
        batch_size=BATCH,
        subjects_per_dataset=6
    )

    train_loader         = DataLoader(train_ds_joint, batch_sampler=sampler)
    train_eval_loader_nb = DataLoader(train_ds_nb, batch_size=256, shuffle=False)
    train_eval_loader_htc = DataLoader(train_ds_htc, batch_size=256, shuffle=False)
    test_loader_nb       = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc      = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_score = -1.0
    best_state = None
    best_acc_nb = -1.0
    best_acc_htc = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, loss_metric, loss_con, loss_proto = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL: NB
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(model, train_eval_loader_nb, device)
            e_test_nb,  y_test_nb_full  = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0) for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )

            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            # =========================
            # ZERO-SHOT EVAL: HTC
            # =========================
            e_train_htc, y_train_htc_full = extract_embeddings(model, train_eval_loader_htc, device)
            e_test_htc,  y_test_htc_full  = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0) for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )

            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_acc_nb = acc_nb
            best_acc_htc = acc_htc

        print(f"Epoch {ep:02d} | NB zero-shot {acc_nb:.4f} | HTC zero-shot {acc_htc:.4f} | avg {score:.4f}")

    # =====================================================
    # LOAD BEST BASE MODEL
    # =====================================================
    model.load_state_dict(best_state)
    best_joint_state = copy.deepcopy(best_state)

    # =====================================================
    # SHIFT COMPUTATION ON BEST BASE MODEL
    # =====================================================
    model.eval()
    with torch.no_grad():
        e_train_nb_base, y_train_nb_base = extract_embeddings(model, train_eval_loader_nb, device)
        e_test_nb_base,  y_test_nb_base  = extract_embeddings(model, test_loader_nb, device)

        e_train_htc_base, y_train_htc_base = extract_embeddings(model, train_eval_loader_htc, device)
        e_test_htc_base,  y_test_htc_base  = extract_embeddings(model, test_loader_htc, device)

    shift_nb = torch.norm(e_train_nb_base.mean(0) - e_test_nb_base.mean(0)).item()
    shift_htc = torch.norm(e_train_htc_base.mean(0) - e_test_htc_base.mean(0)).item()

    print(f"Best-model shift | NB: {shift_nb:.4f} | HTC: {shift_htc:.4f}")

    # =====================================================
    # FEW-SHOT: N-BACK (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    # start NB adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_nb = e_test_nb_base.clone()
    all_y_nb = y_test_nb_base.clone()

    support_idx_nb = []
    for c in torch.unique(all_y_nb):
        class_idx = (all_y_nb == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_nb.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_nb = torch.cat(support_idx_nb)

    mask_nb = torch.zeros(len(all_y_nb), dtype=torch.bool)
    mask_nb[support_idx_nb] = True

    support_subset_nb = torch.utils.data.Subset(test_ds_nb, support_idx_nb.tolist())
    support_loader_nb = DataLoader(support_subset_nb, batch_size=256, shuffle=False)

    use_bn_nb = shift_nb > SHIFT_THRESH
    print(f"N-back BN USED: {use_bn_nb}")

    if use_bn_nb:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_nb:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_nb, all_y_nb = extract_embeddings(model, test_loader_nb, device)

    e_support_nb = all_e_nb[mask_nb].to(device)
    y_support_nb = all_y_nb[mask_nb].to(device)
    e_query_nb   = all_e_nb[~mask_nb].to(device)
    y_query_nb   = all_y_nb[~mask_nb].to(device)

    e_support_nb = F.normalize(e_support_nb, dim=1)
    e_query_nb   = F.normalize(e_query_nb, dim=1)

    var_nb = e_support_nb.var(dim=0)
    weights_nb = 1.0 / (var_nb + 1e-6)
    weights_nb = weights_nb / weights_nb.mean()
    e_support_nb = e_support_nb * weights_nb
    e_query_nb   = e_query_nb * weights_nb

    classes_nb = torch.unique(y_support_nb)
    protos_nb = torch.stack([
        e_support_nb[y_support_nb == c].mean(dim=0) for c in classes_nb
    ])
    protos_nb = F.normalize(protos_nb, dim=1)

    residuals_nb = torch.cat([
        e_support_nb[y_support_nb == c] - protos_nb[i]
        for i, c in enumerate(classes_nb)
    ], dim=0)

    D_nb = residuals_nb.shape[1]
    cov_nb = (residuals_nb.t() @ residuals_nb) / max(residuals_nb.shape[0] - 1, 1)

    avg_var_nb = torch.trace(cov_nb) / D_nb
    cov_shrink_nb = (1 - SHRINK) * cov_nb + SHRINK * (avg_var_nb * torch.eye(D_nb, device=device))
    cov_inv_nb = torch.linalg.pinv(cov_shrink_nb)

    diffs_nb = e_query_nb.unsqueeze(1) - protos_nb.unsqueeze(0)
    dists_nb = torch.einsum("nkd,dd,nkd->nk", diffs_nb, cov_inv_nb, diffs_nb)

    pred_nb_fs = dists_nb.argmin(dim=1)
    fewshot_acc_nb = (pred_nb_fs == y_query_nb).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_nb)

    # =====================================================
    # FEW-SHOT: HTC (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(2234 + fold_i)
    np.random.seed(2234 + fold_i)

    # start HTC adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_htc = e_test_htc_base.clone()
    all_y_htc = y_test_htc_base.clone()

    support_idx_htc = []
    for c in torch.unique(all_y_htc):
        class_idx = (all_y_htc == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_htc.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_htc = torch.cat(support_idx_htc)

    mask_htc = torch.zeros(len(all_y_htc), dtype=torch.bool)
    mask_htc[support_idx_htc] = True

    support_subset_htc = torch.utils.data.Subset(test_ds_htc, support_idx_htc.tolist())
    support_loader_htc = DataLoader(support_subset_htc, batch_size=256, shuffle=False)

    use_bn_htc = shift_htc > SHIFT_THRESH
    print(f"HTC BN USED: {use_bn_htc}")

    if use_bn_htc:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_htc:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_htc, all_y_htc = extract_embeddings(model, test_loader_htc, device)

    e_support_htc = all_e_htc[mask_htc].to(device)
    y_support_htc = all_y_htc[mask_htc].to(device)
    e_query_htc   = all_e_htc[~mask_htc].to(device)
    y_query_htc   = all_y_htc[~mask_htc].to(device)

    e_support_htc = F.normalize(e_support_htc, dim=1)
    e_query_htc   = F.normalize(e_query_htc, dim=1)

    var_htc = e_support_htc.var(dim=0)
    weights_htc = 1.0 / (var_htc + 1e-6)
    weights_htc = weights_htc / weights_htc.mean()
    e_support_htc = e_support_htc * weights_htc
    e_query_htc   = e_query_htc * weights_htc

    classes_htc = torch.unique(y_support_htc)
    protos_htc = torch.stack([
        e_support_htc[y_support_htc == c].mean(dim=0) for c in classes_htc
    ])
    protos_htc = F.normalize(protos_htc, dim=1)

    residuals_htc = torch.cat([
        e_support_htc[y_support_htc == c] - protos_htc[i]
        for i, c in enumerate(classes_htc)
    ], dim=0)

    D_htc = residuals_htc.shape[1]
    cov_htc = (residuals_htc.t() @ residuals_htc) / max(residuals_htc.shape[0] - 1, 1)

    avg_var_htc = torch.trace(cov_htc) / D_htc
    cov_shrink_htc = (1 - SHRINK) * cov_htc + SHRINK * (avg_var_htc * torch.eye(D_htc, device=device))
    cov_inv_htc = torch.linalg.pinv(cov_shrink_htc)

    diffs_htc = e_query_htc.unsqueeze(1) - protos_htc.unsqueeze(0)
    dists_htc = torch.einsum("nkd,dd,nkd->nk", diffs_htc, cov_inv_htc, diffs_htc)

    pred_htc_fs = dists_htc.argmin(dim=1)
    fewshot_acc_htc = (pred_htc_fs == y_query_htc).float().mean().item()

    print("HTC gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_htc)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": best_acc_nb,
        "best_zero_shot_htc": best_acc_htc,
        "shift_nb": shift_nb,
        "shift_htc": shift_htc,
        "bn_used_nb": bool(use_bn_nb),
        "bn_used_htc": bool(use_bn_htc),
        "fewshot_nb": fewshot_acc_nb,
        "fewshot_htc": fewshot_acc_htc
    })

print("\nJOINT METRIC TRAIN + GATED ADABN + MAHALANOBIS LOSO SUMMARY")
for r in overall_results_joint:
    print(
        f"Fold {r['fold']:02d} | "
        f"NB test {r['test_subj_nb']} | HTC test {r['test_subj_htc']} | "
        f"NB zero-shot {r['best_zero_shot_nb']:.4f} | "
        f"HTC zero-shot {r['best_zero_shot_htc']:.4f} | "
        f"NB shift {r['shift_nb']:.4f} | HTC shift {r['shift_htc']:.4f} | "
        f"NB BN {r['bn_used_nb']} | HTC BN {r['bn_used_htc']} | "
        f"NB few-shot {r['fewshot_nb']:.4f} | "
        f"HTC few-shot {r['fewshot_htc']:.4f}"
    )

nb_fs_accs  = [r["fewshot_nb"] for r in overall_results_joint]
htc_fs_accs = [r["fewshot_htc"] for r in overall_results_joint]

print("\nNB few-shot mean/std:", float(np.mean(nb_fs_accs)), float(np.std(nb_fs_accs)))
print("HTC few-shot mean/std:", float(np.mean(htc_fs_accs)), float(np.std(htc_fs_accs)))


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Epoch 01 | NB zero-shot 0.6341 | HTC zero-shot 0.5375 | avg 0.5858
Epoch 02 | NB zero-shot 0.6709 | HTC zero-shot 0.6710 | avg 0.6709
Epoch 03 | NB zero-shot 0.6982 | HTC zero-shot 0.6156 | avg 0.6569
Epoch 04 | NB zero-shot 0.7077 | HTC zero-shot 0.6482 | avg 0.6779
Epoch 05 | NB zero-shot 0.7077 | HTC zero-shot 0.5570 | avg 0.6323
Epoch 06 | NB zero-shot 0.7319 | HTC zero-shot 0.5928 | avg 0.6623
Epoch 07 | NB zero-shot 0.7371 | HTC zero-shot 0.5407 | avg 0.6389
Epoch 08 | NB zero-shot 0.7613 | HTC zero-shot 0.5733 | avg 0.6673
Epoch 09 | NB zero-shot 0.7277 | HTC zero-shot 0.5765 | avg 0.6521
Epoch 10 | NB zero-shot 0.7371 | HTC zero-shot 0.5244 | avg 0.6308
Best-model shift | NB: 0.8757 | HTC: 8.3267
N-back BN USED: True
N-back gated AdaBN + Mahalanobis few-shot acc: 0.7699214816093445
HTC BN USED: True
HTC gated AdaBN + Mahalanobis few-shot acc: 0.8277153372764587

JOINT FOLD 2/13
N-back test subject:

In [ ]:
def mixed_dataset_metric_loss(
    e, proj, yb, sb, db,
    lambda_con=0.5,
    proto_lambda=0.10,
    proj_align_lambda=0.05,   # 🔥 new
    temperature=0.1
):
    e = F.normalize(e, dim=1)
    proj = F.normalize(proj, dim=1)

    loss_metric = torch.tensor(0.0, device=e.device)
    loss_proto  = torch.tensor(0.0, device=e.device)

    mask_nb  = (db == NBACK_ID)
    mask_htc = (db == HTC_ID)

    # ----------------------------
    # PROTOTYPE LOSSES (on e)
    # ----------------------------
    if mask_nb.any():
        loss_metric += prototype_ce_loss(e[mask_nb], yb[mask_nb], temperature)
        loss_proto  += prototype_loss(e[mask_nb], yb[mask_nb])

    if mask_htc.any():
        loss_metric += prototype_ce_loss(e[mask_htc], yb[mask_htc], temperature)
        loss_proto  += prototype_loss(e[mask_htc], yb[mask_htc])

    # ----------------------------
    # SUPCON (back to proj)
    # ----------------------------
    loss_con = supcon_within_dataset_loss(
        proj, yb, sb, db, temperature=temperature
    )

    # ----------------------------
    # 🔥 NEW: ALIGN proj → e
    # ----------------------------
    loss_align = F.mse_loss(proj, e)

    # ----------------------------
    # FINAL
    # ----------------------------
    total = (
        loss_metric
        + lambda_con * loss_con
        + proto_lambda * loss_proto
        + proj_align_lambda * loss_align
    )

    return total, loss_metric, loss_con, loss_proto

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))
overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLIT
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb  = np.where(train_mask_nb)[0]
    test_idx_nb   = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc  = np.where(train_mask_htc)[0]
    test_idx_htc   = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0, 2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0, 2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    sampler = JointBalancedBatchSampler(
        train_ds_joint,
        batch_size=BATCH,
        subjects_per_dataset=6
    )

    train_loader         = DataLoader(train_ds_joint, batch_sampler=sampler)
    train_eval_loader_nb = DataLoader(train_ds_nb, batch_size=256, shuffle=False)
    train_eval_loader_htc = DataLoader(train_ds_htc, batch_size=256, shuffle=False)
    test_loader_nb       = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc      = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_score = -1.0
    best_state = None
    best_acc_nb = -1.0
    best_acc_htc = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, loss_metric, loss_con, loss_proto = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL: NB
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(model, train_eval_loader_nb, device)
            e_test_nb,  y_test_nb_full  = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0) for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )

            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            # =========================
            # ZERO-SHOT EVAL: HTC
            # =========================
            e_train_htc, y_train_htc_full = extract_embeddings(model, train_eval_loader_htc, device)
            e_test_htc,  y_test_htc_full  = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0) for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )

            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_acc_nb = acc_nb
            best_acc_htc = acc_htc

        print(f"Epoch {ep:02d} | NB zero-shot {acc_nb:.4f} | HTC zero-shot {acc_htc:.4f} | avg {score:.4f}")

    # =====================================================
    # LOAD BEST BASE MODEL
    # =====================================================
    model.load_state_dict(best_state)
    best_joint_state = copy.deepcopy(best_state)

    # =====================================================
    # SHIFT COMPUTATION ON BEST BASE MODEL
    # =====================================================
    model.eval()
    with torch.no_grad():
        e_train_nb_base, y_train_nb_base = extract_embeddings(model, train_eval_loader_nb, device)
        e_test_nb_base,  y_test_nb_base  = extract_embeddings(model, test_loader_nb, device)

        e_train_htc_base, y_train_htc_base = extract_embeddings(model, train_eval_loader_htc, device)
        e_test_htc_base,  y_test_htc_base  = extract_embeddings(model, test_loader_htc, device)

    shift_nb = torch.norm(e_train_nb_base.mean(0) - e_test_nb_base.mean(0)).item()
    shift_htc = torch.norm(e_train_htc_base.mean(0) - e_test_htc_base.mean(0)).item()

    print(f"Best-model shift | NB: {shift_nb:.4f} | HTC: {shift_htc:.4f}")

    # =====================================================
    # FEW-SHOT: N-BACK (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    # start NB adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_nb = e_test_nb_base.clone()
    all_y_nb = y_test_nb_base.clone()

    support_idx_nb = []
    for c in torch.unique(all_y_nb):
        class_idx = (all_y_nb == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_nb.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_nb = torch.cat(support_idx_nb)

    mask_nb = torch.zeros(len(all_y_nb), dtype=torch.bool)
    mask_nb[support_idx_nb] = True

    support_subset_nb = torch.utils.data.Subset(test_ds_nb, support_idx_nb.tolist())
    support_loader_nb = DataLoader(support_subset_nb, batch_size=256, shuffle=False)

    use_bn_nb = shift_nb > SHIFT_THRESH
    print(f"N-back BN USED: {use_bn_nb}")

    if use_bn_nb:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_nb:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_nb, all_y_nb = extract_embeddings(model, test_loader_nb, device)

    e_support_nb = all_e_nb[mask_nb].to(device)
    y_support_nb = all_y_nb[mask_nb].to(device)
    e_query_nb   = all_e_nb[~mask_nb].to(device)
    y_query_nb   = all_y_nb[~mask_nb].to(device)

    e_support_nb = F.normalize(e_support_nb, dim=1)
    e_query_nb   = F.normalize(e_query_nb, dim=1)

    var_nb = e_support_nb.var(dim=0)
    weights_nb = 1.0 / (var_nb + 1e-6)
    weights_nb = weights_nb / weights_nb.mean()
    e_support_nb = e_support_nb * weights_nb
    e_query_nb   = e_query_nb * weights_nb

    classes_nb = torch.unique(y_support_nb)
    protos_nb = torch.stack([
        e_support_nb[y_support_nb == c].mean(dim=0) for c in classes_nb
    ])
    protos_nb = F.normalize(protos_nb, dim=1)

    residuals_nb = torch.cat([
        e_support_nb[y_support_nb == c] - protos_nb[i]
        for i, c in enumerate(classes_nb)
    ], dim=0)

    D_nb = residuals_nb.shape[1]
    cov_nb = (residuals_nb.t() @ residuals_nb) / max(residuals_nb.shape[0] - 1, 1)

    avg_var_nb = torch.trace(cov_nb) / D_nb
    cov_shrink_nb = (1 - SHRINK) * cov_nb + SHRINK * (avg_var_nb * torch.eye(D_nb, device=device))
    cov_inv_nb = torch.linalg.pinv(cov_shrink_nb)

    diffs_nb = e_query_nb.unsqueeze(1) - protos_nb.unsqueeze(0)
    dists_nb = torch.einsum("nkd,dd,nkd->nk", diffs_nb, cov_inv_nb, diffs_nb)

    pred_nb_fs = dists_nb.argmin(dim=1)
    fewshot_acc_nb = (pred_nb_fs == y_query_nb).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_nb)

    # =====================================================
    # FEW-SHOT: HTC (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(2234 + fold_i)
    np.random.seed(2234 + fold_i)

    # start HTC adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_htc = e_test_htc_base.clone()
    all_y_htc = y_test_htc_base.clone()

    support_idx_htc = []
    for c in torch.unique(all_y_htc):
        class_idx = (all_y_htc == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_htc.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_htc = torch.cat(support_idx_htc)

    mask_htc = torch.zeros(len(all_y_htc), dtype=torch.bool)
    mask_htc[support_idx_htc] = True

    support_subset_htc = torch.utils.data.Subset(test_ds_htc, support_idx_htc.tolist())
    support_loader_htc = DataLoader(support_subset_htc, batch_size=256, shuffle=False)

    use_bn_htc = shift_htc > SHIFT_THRESH
    print(f"HTC BN USED: {use_bn_htc}")

    if use_bn_htc:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_htc:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_htc, all_y_htc = extract_embeddings(model, test_loader_htc, device)

    e_support_htc = all_e_htc[mask_htc].to(device)
    y_support_htc = all_y_htc[mask_htc].to(device)
    e_query_htc   = all_e_htc[~mask_htc].to(device)
    y_query_htc   = all_y_htc[~mask_htc].to(device)

    e_support_htc = F.normalize(e_support_htc, dim=1)
    e_query_htc   = F.normalize(e_query_htc, dim=1)

    var_htc = e_support_htc.var(dim=0)
    weights_htc = 1.0 / (var_htc + 1e-6)
    weights_htc = weights_htc / weights_htc.mean()
    e_support_htc = e_support_htc * weights_htc
    e_query_htc   = e_query_htc * weights_htc

    classes_htc = torch.unique(y_support_htc)
    protos_htc = torch.stack([
        e_support_htc[y_support_htc == c].mean(dim=0) for c in classes_htc
    ])
    protos_htc = F.normalize(protos_htc, dim=1)

    residuals_htc = torch.cat([
        e_support_htc[y_support_htc == c] - protos_htc[i]
        for i, c in enumerate(classes_htc)
    ], dim=0)

    D_htc = residuals_htc.shape[1]
    cov_htc = (residuals_htc.t() @ residuals_htc) / max(residuals_htc.shape[0] - 1, 1)

    avg_var_htc = torch.trace(cov_htc) / D_htc
    cov_shrink_htc = (1 - SHRINK) * cov_htc + SHRINK * (avg_var_htc * torch.eye(D_htc, device=device))
    cov_inv_htc = torch.linalg.pinv(cov_shrink_htc)

    diffs_htc = e_query_htc.unsqueeze(1) - protos_htc.unsqueeze(0)
    dists_htc = torch.einsum("nkd,dd,nkd->nk", diffs_htc, cov_inv_htc, diffs_htc)

    pred_htc_fs = dists_htc.argmin(dim=1)
    fewshot_acc_htc = (pred_htc_fs == y_query_htc).float().mean().item()

    print("HTC gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_htc)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": best_acc_nb,
        "best_zero_shot_htc": best_acc_htc,
        "shift_nb": shift_nb,
        "shift_htc": shift_htc,
        "bn_used_nb": bool(use_bn_nb),
        "bn_used_htc": bool(use_bn_htc),
        "fewshot_nb": fewshot_acc_nb,
        "fewshot_htc": fewshot_acc_htc
    })

print("\nJOINT METRIC TRAIN + GATED ADABN + MAHALANOBIS LOSO SUMMARY")
for r in overall_results_joint:
    print(
        f"Fold {r['fold']:02d} | "
        f"NB test {r['test_subj_nb']} | HTC test {r['test_subj_htc']} | "
        f"NB zero-shot {r['best_zero_shot_nb']:.4f} | "
        f"HTC zero-shot {r['best_zero_shot_htc']:.4f} | "
        f"NB shift {r['shift_nb']:.4f} | HTC shift {r['shift_htc']:.4f} | "
        f"NB BN {r['bn_used_nb']} | HTC BN {r['bn_used_htc']} | "
        f"NB few-shot {r['fewshot_nb']:.4f} | "
        f"HTC few-shot {r['fewshot_htc']:.4f}"
    )

nb_fs_accs  = [r["fewshot_nb"] for r in overall_results_joint]
htc_fs_accs = [r["fewshot_htc"] for r in overall_results_joint]

print("\nNB few-shot mean/std:", float(np.mean(nb_fs_accs)), float(np.std(nb_fs_accs)))
print("HTC few-shot mean/std:", float(np.mean(htc_fs_accs)), float(np.std(htc_fs_accs)))


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Epoch 01 | NB zero-shot 0.6215 | HTC zero-shot 0.6352 | avg 0.6283
Epoch 02 | NB zero-shot 0.6772 | HTC zero-shot 0.6384 | avg 0.6578
Epoch 03 | NB zero-shot 0.6719 | HTC zero-shot 0.6156 | avg 0.6438
Epoch 04 | NB zero-shot 0.6930 | HTC zero-shot 0.5505 | avg 0.6217
Epoch 05 | NB zero-shot 0.7098 | HTC zero-shot 0.6124 | avg 0.6611
Epoch 06 | NB zero-shot 0.7340 | HTC zero-shot 0.5668 | avg 0.6504
Epoch 07 | NB zero-shot 0.7350 | HTC zero-shot 0.5961 | avg 0.6656
Epoch 08 | NB zero-shot 0.7192 | HTC zero-shot 0.5668 | avg 0.6430
Epoch 09 | NB zero-shot 0.6982 | HTC zero-shot 0.5603 | avg 0.6292
Epoch 10 | NB zero-shot 0.7581 | HTC zero-shot 0.5212 | avg 0.6397
Best-model shift | NB: 0.9392 | HTC: 1.9574
N-back BN USED: True
N-back gated AdaBN + Mahalanobis few-shot acc: 0.7946128249168396
HTC BN USED: True
HTC gated AdaBN + Mahalanobis few-shot acc: 0.6928839087486267

JOINT FOLD 2/13
N-back test subject:

In [ ]:
'''
💥 WHAT SUPCON WAS DOING (IMPORTANT)

Even when it was “imperfect,” SupCon was:

forcing same-label samples across subjects to align

👉 THAT is what helps:

cross-subject generalization
few-shot performance
robustness
🔥 NOW YOU REMOVED THAT

So your model becomes:

over-structured but under-regularized
'''

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class FusionEncoder(nn.Module):
#     def __init__(self, emb_dim=128, bp_dim=42, roi_dim=3):
#         super().__init__()

#         total_bp_dim = bp_dim + roi_dim

#         # ------------------
#         # TIME BRANCH (FIXED)
#         # ------------------
#         self.time_branch = nn.Sequential(
#             nn.Conv1d(14, 32, kernel_size=7, padding=3),
#             nn.BatchNorm1d(32),
#             nn.GELU(),

#             nn.Conv1d(32, 64, kernel_size=5, padding=2),
#             nn.BatchNorm1d(64),
#             nn.GELU(),

#             nn.Conv1d(64, 64, kernel_size=3, padding=1),
#             nn.BatchNorm1d(64),
#             nn.GELU(),
#         )

#         self.channel_attn = nn.Sequential(
#             nn.AdaptiveAvgPool1d(1),
#             nn.Conv1d(64, 16, kernel_size=1),
#             nn.GELU(),
#             nn.Conv1d(16, 64, kernel_size=1),
#             nn.Sigmoid()
#         )

#         self.attn_pool = nn.Conv1d(64, 1, kernel_size=1)

#         # ------------------
#         # BANDPOWER
#         # ------------------
#         self.bp_branch = nn.Sequential(
#             nn.Linear(total_bp_dim, 128),
#             nn.BatchNorm1d(128),
#             nn.GELU(),
#             nn.Dropout(0.3),
#             nn.Linear(128, 64)
#         )

#         # ------------------
#         # COVARIANCE
#         # ------------------
#         self.cov_branch = nn.Sequential(
#             nn.Linear(105, 256),
#             nn.BatchNorm1d(256),
#             nn.GELU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, 64)
#         )

#         # ------------------
#         # PLV
#         # ------------------
#         self.plv_branch = nn.Sequential(
#             nn.Linear(210, 256),
#             nn.BatchNorm1d(256),
#             nn.GELU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, 64)
#         )

#         # ------------------
#         # FUSION (DEEPER)
#         # ------------------
#         self.embed = nn.Sequential(
#             nn.Linear(64 + 64 + 64 + 64, 256),
#             nn.BatchNorm1d(256),
#             nn.GELU(),
#             nn.Dropout(0.4),

#             nn.Linear(256, 128),
#             nn.BatchNorm1d(128),
#             nn.GELU(),

#             nn.Linear(128, emb_dim)
#         )

#     def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):

#         # TIME
#         x = self.time_branch(x_time)
#         x = x * self.channel_attn(x)

#         weights = F.softmax(self.attn_pool(x), dim=-1)
#         z_time = (x * weights).sum(dim=-1)

#         # OTHER BRANCHES
#         z_bp  = self.bp_branch(torch.cat([x_bp, x_roi], dim=1))
#         z_cov = self.cov_branch(x_cov)
#         z_plv = self.plv_branch(x_plv)

#         # FUSION
#         z = torch.cat([z_time, z_bp, z_cov, z_plv], dim=1)
#         e = self.embed(z)

#         return e

In [ ]:
import torch
import torch.nn.functional as F

def mixed_dataset_metric_loss(
    e, proj, yb, sb, db,
    lambda_con=0.5,
    proto_lambda=0.10,
    temperature=0.1
):
    # ----------------------------
    # normalize embeddings
    # ----------------------------
    e = F.normalize(e, dim=1)
    proj = F.normalize(proj, dim=1)

    loss_metric = torch.tensor(0.0, device=e.device)
    loss_proto  = torch.tensor(0.0, device=e.device)

    mask_nb  = (db == NBACK_ID)
    mask_htc = (db == HTC_ID)

    # ----------------------------
    # PROTOTYPE + METRIC
    # ----------------------------
    if mask_nb.any():
        loss_metric += prototype_ce_loss(
            e[mask_nb], yb[mask_nb], temperature=temperature
        )
        loss_proto += prototype_loss(
            e[mask_nb], yb[mask_nb]
        )

    if mask_htc.any():
        loss_metric += prototype_ce_loss(
            e[mask_htc], yb[mask_htc], temperature=temperature
        )
        loss_proto += prototype_loss(
            e[mask_htc], yb[mask_htc]
        )

    # ----------------------------
    # SUPCON
    # ----------------------------
    loss_con = supcon_within_dataset_loss(
        proj, yb, sb, db, temperature=temperature
    )

    # ----------------------------
    # 🔥 FIXED: DISTANCE CONSISTENCY
    # ----------------------------
    loss_dist = torch.tensor(0.0, device=e.device)

    if mask_nb.any() and mask_htc.any():

        def get_dist_vector(e_sub, y_sub):
            classes = torch.unique(y_sub)

            if len(classes) < 2:
                return None

            protos = torch.stack([
                e_sub[y_sub == c].mean(dim=0)
                for c in classes
            ])

            protos = F.normalize(protos, dim=1)

            # pairwise distances
            dists = torch.cdist(protos, protos, p=2)

            # 🔥 take upper triangle only (ignore size mismatch)
            idx = torch.triu_indices(dists.size(0), dists.size(1), offset=1)
            vec = dists[idx[0], idx[1]]

            return vec

        d_nb  = get_dist_vector(e[mask_nb], yb[mask_nb])
        d_htc = get_dist_vector(e[mask_htc], yb[mask_htc])

        if d_nb is not None and d_htc is not None:

            # normalize
            d_nb  = d_nb  / (d_nb.mean()  + 1e-6)
            d_htc = d_htc / (d_htc.mean() + 1e-6)

            # 🔥 match only overlapping length
            L = min(len(d_nb), len(d_htc))

            if L > 0:
                loss_dist = F.mse_loss(d_nb[:L], d_htc[:L])

    # ----------------------------
    # FINAL LOSS
    # ----------------------------
    total = (
        loss_metric
        + lambda_con * loss_con
        + proto_lambda * loss_proto
        + 0.1 * loss_dist
    )

    return total, loss_metric, loss_con, loss_proto

In [ ]:
import torch.nn as nn

class ContrastiveModel(nn.Module):
    def __init__(self,
                 emb_dim=64,
                 proj_dim=64,
                 bp_dim=42,
                 roi_dim=3):

        super().__init__()

        # Main encoder (learns embedding geometry)
        self.encoder = FusionEncoder(
            emb_dim=emb_dim,
            bp_dim=bp_dim,
            roi_dim=roi_dim
        )

        # Projection head for SupCon loss
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):
        # Embedding
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)

        # Projection (used only for contrastive loss)
        p = self.proj(e)

        return e, p

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))
overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLIT
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb  = np.where(train_mask_nb)[0]
    test_idx_nb   = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc  = np.where(train_mask_htc)[0]
    test_idx_htc   = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0, 2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0, 2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    sampler = JointBalancedBatchSampler(
        train_ds_joint,
        batch_size=BATCH,
        subjects_per_dataset=6
    )

    train_loader         = DataLoader(train_ds_joint, batch_sampler=sampler)
    train_eval_loader_nb = DataLoader(train_ds_nb, batch_size=256, shuffle=False)
    train_eval_loader_htc = DataLoader(train_ds_htc, batch_size=256, shuffle=False)
    test_loader_nb       = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc      = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_score = -1.0
    best_state = None
    best_acc_nb = -1.0
    best_acc_htc = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, loss_metric, loss_con, loss_proto = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL: NB
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(model, train_eval_loader_nb, device)
            e_test_nb,  y_test_nb_full  = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0) for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )

            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            # =========================
            # ZERO-SHOT EVAL: HTC
            # =========================
            e_train_htc, y_train_htc_full = extract_embeddings(model, train_eval_loader_htc, device)
            e_test_htc,  y_test_htc_full  = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0) for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )

            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_acc_nb = acc_nb
            best_acc_htc = acc_htc

        print(f"Epoch {ep:02d} | NB zero-shot {acc_nb:.4f} | HTC zero-shot {acc_htc:.4f} | avg {score:.4f}")

    # =====================================================
    # LOAD BEST BASE MODEL
    # =====================================================
    model.load_state_dict(best_state)
    best_joint_state = copy.deepcopy(best_state)

    # =====================================================
    # SHIFT COMPUTATION ON BEST BASE MODEL
    # =====================================================
    model.eval()
    with torch.no_grad():
        e_train_nb_base, y_train_nb_base = extract_embeddings(model, train_eval_loader_nb, device)
        e_test_nb_base,  y_test_nb_base  = extract_embeddings(model, test_loader_nb, device)

        e_train_htc_base, y_train_htc_base = extract_embeddings(model, train_eval_loader_htc, device)
        e_test_htc_base,  y_test_htc_base  = extract_embeddings(model, test_loader_htc, device)

    shift_nb = torch.norm(e_train_nb_base.mean(0) - e_test_nb_base.mean(0)).item()
    shift_htc = torch.norm(e_train_htc_base.mean(0) - e_test_htc_base.mean(0)).item()

    print(f"Best-model shift | NB: {shift_nb:.4f} | HTC: {shift_htc:.4f}")

    # =====================================================
    # FEW-SHOT: N-BACK (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    # start NB adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_nb = e_test_nb_base.clone()
    all_y_nb = y_test_nb_base.clone()

    support_idx_nb = []
    for c in torch.unique(all_y_nb):
        class_idx = (all_y_nb == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_nb.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_nb = torch.cat(support_idx_nb)

    mask_nb = torch.zeros(len(all_y_nb), dtype=torch.bool)
    mask_nb[support_idx_nb] = True

    support_subset_nb = torch.utils.data.Subset(test_ds_nb, support_idx_nb.tolist())
    support_loader_nb = DataLoader(support_subset_nb, batch_size=256, shuffle=False)

    use_bn_nb = shift_nb > SHIFT_THRESH
    print(f"N-back BN USED: {use_bn_nb}")

    if use_bn_nb:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_nb:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_nb, all_y_nb = extract_embeddings(model, test_loader_nb, device)

    e_support_nb = all_e_nb[mask_nb].to(device)
    y_support_nb = all_y_nb[mask_nb].to(device)
    e_query_nb   = all_e_nb[~mask_nb].to(device)
    y_query_nb   = all_y_nb[~mask_nb].to(device)

    e_support_nb = F.normalize(e_support_nb, dim=1)
    e_query_nb   = F.normalize(e_query_nb, dim=1)

    var_nb = e_support_nb.var(dim=0)
    weights_nb = 1.0 / (var_nb + 1e-6)
    weights_nb = weights_nb / weights_nb.mean()
    e_support_nb = e_support_nb * weights_nb
    e_query_nb   = e_query_nb * weights_nb

    classes_nb = torch.unique(y_support_nb)
    protos_nb = torch.stack([
        e_support_nb[y_support_nb == c].mean(dim=0) for c in classes_nb
    ])
    protos_nb = F.normalize(protos_nb, dim=1)

    residuals_nb = torch.cat([
        e_support_nb[y_support_nb == c] - protos_nb[i]
        for i, c in enumerate(classes_nb)
    ], dim=0)

    D_nb = residuals_nb.shape[1]
    cov_nb = (residuals_nb.t() @ residuals_nb) / max(residuals_nb.shape[0] - 1, 1)

    avg_var_nb = torch.trace(cov_nb) / D_nb
    cov_shrink_nb = (1 - SHRINK) * cov_nb + SHRINK * (avg_var_nb * torch.eye(D_nb, device=device))
    cov_inv_nb = torch.linalg.pinv(cov_shrink_nb)

    diffs_nb = e_query_nb.unsqueeze(1) - protos_nb.unsqueeze(0)
    dists_nb = torch.einsum("nkd,dd,nkd->nk", diffs_nb, cov_inv_nb, diffs_nb)

    pred_nb_fs = dists_nb.argmin(dim=1)
    fewshot_acc_nb = (pred_nb_fs == y_query_nb).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_nb)

    # =====================================================
    # FEW-SHOT: HTC (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(2234 + fold_i)
    np.random.seed(2234 + fold_i)

    # start HTC adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_htc = e_test_htc_base.clone()
    all_y_htc = y_test_htc_base.clone()

    support_idx_htc = []
    for c in torch.unique(all_y_htc):
        class_idx = (all_y_htc == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_htc.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_htc = torch.cat(support_idx_htc)

    mask_htc = torch.zeros(len(all_y_htc), dtype=torch.bool)
    mask_htc[support_idx_htc] = True

    support_subset_htc = torch.utils.data.Subset(test_ds_htc, support_idx_htc.tolist())
    support_loader_htc = DataLoader(support_subset_htc, batch_size=256, shuffle=False)

    use_bn_htc = shift_htc > SHIFT_THRESH
    print(f"HTC BN USED: {use_bn_htc}")

    if use_bn_htc:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_htc:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_htc, all_y_htc = extract_embeddings(model, test_loader_htc, device)

    e_support_htc = all_e_htc[mask_htc].to(device)
    y_support_htc = all_y_htc[mask_htc].to(device)
    e_query_htc   = all_e_htc[~mask_htc].to(device)
    y_query_htc   = all_y_htc[~mask_htc].to(device)

    e_support_htc = F.normalize(e_support_htc, dim=1)
    e_query_htc   = F.normalize(e_query_htc, dim=1)

    var_htc = e_support_htc.var(dim=0)
    weights_htc = 1.0 / (var_htc + 1e-6)
    weights_htc = weights_htc / weights_htc.mean()
    e_support_htc = e_support_htc * weights_htc
    e_query_htc   = e_query_htc * weights_htc

    classes_htc = torch.unique(y_support_htc)
    protos_htc = torch.stack([
        e_support_htc[y_support_htc == c].mean(dim=0) for c in classes_htc
    ])
    protos_htc = F.normalize(protos_htc, dim=1)

    residuals_htc = torch.cat([
        e_support_htc[y_support_htc == c] - protos_htc[i]
        for i, c in enumerate(classes_htc)
    ], dim=0)

    D_htc = residuals_htc.shape[1]
    cov_htc = (residuals_htc.t() @ residuals_htc) / max(residuals_htc.shape[0] - 1, 1)

    avg_var_htc = torch.trace(cov_htc) / D_htc
    cov_shrink_htc = (1 - SHRINK) * cov_htc + SHRINK * (avg_var_htc * torch.eye(D_htc, device=device))
    cov_inv_htc = torch.linalg.pinv(cov_shrink_htc)

    diffs_htc = e_query_htc.unsqueeze(1) - protos_htc.unsqueeze(0)
    dists_htc = torch.einsum("nkd,dd,nkd->nk", diffs_htc, cov_inv_htc, diffs_htc)

    pred_htc_fs = dists_htc.argmin(dim=1)
    fewshot_acc_htc = (pred_htc_fs == y_query_htc).float().mean().item()

    print("HTC gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_htc)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": best_acc_nb,
        "best_zero_shot_htc": best_acc_htc,
        "shift_nb": shift_nb,
        "shift_htc": shift_htc,
        "bn_used_nb": bool(use_bn_nb),
        "bn_used_htc": bool(use_bn_htc),
        "fewshot_nb": fewshot_acc_nb,
        "fewshot_htc": fewshot_acc_htc
    })

print("\nJOINT METRIC TRAIN + GATED ADABN + MAHALANOBIS LOSO SUMMARY")
for r in overall_results_joint:
    print(
        f"Fold {r['fold']:02d} | "
        f"NB test {r['test_subj_nb']} | HTC test {r['test_subj_htc']} | "
        f"NB zero-shot {r['best_zero_shot_nb']:.4f} | "
        f"HTC zero-shot {r['best_zero_shot_htc']:.4f} | "
        f"NB shift {r['shift_nb']:.4f} | HTC shift {r['shift_htc']:.4f} | "
        f"NB BN {r['bn_used_nb']} | HTC BN {r['bn_used_htc']} | "
        f"NB few-shot {r['fewshot_nb']:.4f} | "
        f"HTC few-shot {r['fewshot_htc']:.4f}"
    )

nb_fs_accs  = [r["fewshot_nb"] for r in overall_results_joint]
htc_fs_accs = [r["fewshot_htc"] for r in overall_results_joint]

print("\nNB few-shot mean/std:", float(np.mean(nb_fs_accs)), float(np.std(nb_fs_accs)))
print("HTC few-shot mean/std:", float(np.mean(htc_fs_accs)), float(np.std(htc_fs_accs)))


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Epoch 01 | NB zero-shot 0.6635 | HTC zero-shot 0.6580 | avg 0.6607
Epoch 02 | NB zero-shot 0.7066 | HTC zero-shot 0.5635 | avg 0.6351
Epoch 03 | NB zero-shot 0.6572 | HTC zero-shot 0.5863 | avg 0.6218
Epoch 04 | NB zero-shot 0.6761 | HTC zero-shot 0.5309 | avg 0.6035
Epoch 05 | NB zero-shot 0.7056 | HTC zero-shot 0.5765 | avg 0.6411
Epoch 06 | NB zero-shot 0.6730 | HTC zero-shot 0.5863 | avg 0.6296
Epoch 07 | NB zero-shot 0.6372 | HTC zero-shot 0.6352 | avg 0.6362
Epoch 08 | NB zero-shot 0.6761 | HTC zero-shot 0.5342 | avg 0.6052
Epoch 09 | NB zero-shot 0.6646 | HTC zero-shot 0.5928 | avg 0.6287
Epoch 10 | NB zero-shot 0.6414 | HTC zero-shot 0.5375 | avg 0.5894
Best-model shift | NB: 0.5141 | HTC: 3.2965
N-back BN USED: True
N-back gated AdaBN + Mahalanobis few-shot acc: 0.7665544748306274
HTC BN USED: True
HTC gated AdaBN + Mahalanobis few-shot acc: 0.7303370833396912

JOINT FOLD 2/13
N-back test subject:

In [ ]:
def supcon_cross_dataset_loss(proj, yb, temperature=0.1):
    proj = F.normalize(proj, dim=1)

    sim = torch.matmul(proj, proj.T) / temperature

    labels = yb.unsqueeze(1)
    mask = (labels == labels.T).float()

    logits_mask = torch.ones_like(mask) - torch.eye(len(mask), device=mask.device)
    mask = mask * logits_mask

    exp_sim = torch.exp(sim) * logits_mask
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

    mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-8)

    loss = -mean_log_prob_pos.mean()
    return loss

In [ ]:
import torch
import torch.nn.functional as F

def mixed_dataset_metric_loss(
    e, proj, yb, sb, db,
    lambda_con=0.5,
    proto_lambda=0.10,
    temperature=0.1
):
    # ----------------------------
    # normalize embeddings
    # ----------------------------
    e = F.normalize(e, dim=1)
    proj = F.normalize(proj, dim=1)

    loss_metric = torch.tensor(0.0, device=e.device)
    loss_proto  = torch.tensor(0.0, device=e.device)

    mask_nb  = (db == NBACK_ID)
    mask_htc = (db == HTC_ID)

    # ----------------------------
    # PROTOTYPE + METRIC
    # ----------------------------
    if mask_nb.any():
        loss_metric += prototype_ce_loss(
            e[mask_nb], yb[mask_nb], temperature=temperature
        )
        loss_proto += prototype_loss(
            e[mask_nb], yb[mask_nb]
        )

    if mask_htc.any():
        loss_metric += prototype_ce_loss(
            e[mask_htc], yb[mask_htc], temperature=temperature
        )
        loss_proto += prototype_loss(
            e[mask_htc], yb[mask_htc]
        )

    # ----------------------------
    # SUPCON
    # ----------------------------
    # loss_con = supcon_cross_dataset_loss(
    # proj, yb, temperature=temperature
    # )
    loss_con = supcon_within_dataset_loss(
    proj, yb, sb, db, temperature=temperature
    )

    # ----------------------------
    # 🔥 FIXED: DISTANCE CONSISTENCY
    # ----------------------------
    loss_dist = torch.tensor(0.0, device=e.device)

    if mask_nb.any() and mask_htc.any():

        def get_dist_vector(e_sub, y_sub):
            classes = torch.unique(y_sub)

            if len(classes) < 2:
                return None

            protos = torch.stack([
                e_sub[y_sub == c].mean(dim=0)
                for c in classes
            ])

            protos = F.normalize(protos, dim=1)

            # pairwise distances
            dists = torch.cdist(protos, protos, p=2)

            # 🔥 take upper triangle only (ignore size mismatch)
            idx = torch.triu_indices(dists.size(0), dists.size(1), offset=1)
            vec = dists[idx[0], idx[1]]

            return vec

        d_nb  = get_dist_vector(e[mask_nb], yb[mask_nb])
        d_htc = get_dist_vector(e[mask_htc], yb[mask_htc])

        if d_nb is not None and d_htc is not None:

            # normalize
            d_nb  = d_nb  / (d_nb.mean()  + 1e-6)
            d_htc = d_htc / (d_htc.mean() + 1e-6)

            # 🔥 match only overlapping length
            L = min(len(d_nb), len(d_htc))

            if L > 0:
                loss_dist = F.mse_loss(d_nb[:L], d_htc[:L])

    # ----------------------------
    # FINAL LOSS
    # ----------------------------
    total = (
        loss_metric
        + lambda_con * loss_con
        + proto_lambda * loss_proto
        + 0.1 * loss_dist
    )

    return total, loss_metric, loss_con, loss_proto

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionEncoder(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=3):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim   # 42 + 3 = 45

        # ------------------
        # TIME BRANCH
        # ------------------
        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        # ------------------
        # BANDPOWER + ROI BRANCH
        # ------------------
        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),   # was 42 → now 45
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        # ------------------
        # COVARIANCE BRANCH
        # ------------------
        self.cov_branch = nn.Sequential(
            nn.Linear(105, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # PLV BRANCH
        # ------------------
        self.plv_branch = nn.Sequential(
            nn.Linear(210, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # FUSION
        # ------------------
        self.embed = nn.Sequential(
            nn.Linear(32 + 32 + 32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):

        # ------------------
        # Time branch
        # ------------------
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)  # (B,32)

        # ------------------
        # Bandpower + ROI
        # ------------------
        bp_in = torch.cat([x_bp, x_roi], dim=1)  # (B,45)
        z_bp = self.bp_branch(bp_in)

        # ------------------
        # Covariance
        # ------------------
        z_cov = self.cov_branch(x_cov)

        # ------------------
        # PLV
        # ------------------
        z_plv = self.plv_branch(x_plv)

        # ------------------
        # Fusion
        # ------------------
        z = torch.cat([z_time, z_bp, z_cov, z_plv], dim=1)
        e = self.embed(z)

        return e

In [ ]:
import torch.nn as nn

class ContrastiveModel(nn.Module):
    def __init__(self,
                 emb_dim=64,
                 proj_dim=64,
                 bp_dim=42,
                 roi_dim=3):

        super().__init__()

        # Main encoder (learns embedding geometry)
        self.encoder = FusionEncoder(
            emb_dim=emb_dim,
            bp_dim=bp_dim,
            roi_dim=roi_dim
        )

        # Projection head for SupCon loss
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):
        # Embedding
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)

        # Projection (used only for contrastive loss)
        p = self.proj(e)

        return e, p

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))
overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLIT
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb  = np.where(train_mask_nb)[0]
    test_idx_nb   = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc  = np.where(train_mask_htc)[0]
    test_idx_htc   = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0, 2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0, 2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    sampler = JointBalancedBatchSampler(
        train_ds_joint,
        batch_size=BATCH,
        subjects_per_dataset=6
    )

    train_loader         = DataLoader(train_ds_joint, batch_sampler=sampler)
    train_eval_loader_nb = DataLoader(train_ds_nb, batch_size=256, shuffle=False)
    train_eval_loader_htc = DataLoader(train_ds_htc, batch_size=256, shuffle=False)
    test_loader_nb       = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc      = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_score = -1.0
    best_state = None
    best_acc_nb = -1.0
    best_acc_htc = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, loss_metric, loss_con, loss_proto = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL: NB
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(model, train_eval_loader_nb, device)
            e_test_nb,  y_test_nb_full  = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0) for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )

            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            # =========================
            # ZERO-SHOT EVAL: HTC
            # =========================
            e_train_htc, y_train_htc_full = extract_embeddings(model, train_eval_loader_htc, device)
            e_test_htc,  y_test_htc_full  = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0) for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )

            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_acc_nb = acc_nb
            best_acc_htc = acc_htc

        print(f"Epoch {ep:02d} | NB zero-shot {acc_nb:.4f} | HTC zero-shot {acc_htc:.4f} | avg {score:.4f}")

    # =====================================================
    # LOAD BEST BASE MODEL
    # =====================================================
    model.load_state_dict(best_state)
    best_joint_state = copy.deepcopy(best_state)

    # =====================================================
    # SHIFT COMPUTATION ON BEST BASE MODEL
    # =====================================================
    model.eval()
    with torch.no_grad():
        e_train_nb_base, y_train_nb_base = extract_embeddings(model, train_eval_loader_nb, device)
        e_test_nb_base,  y_test_nb_base  = extract_embeddings(model, test_loader_nb, device)

        e_train_htc_base, y_train_htc_base = extract_embeddings(model, train_eval_loader_htc, device)
        e_test_htc_base,  y_test_htc_base  = extract_embeddings(model, test_loader_htc, device)

    shift_nb = torch.norm(e_train_nb_base.mean(0) - e_test_nb_base.mean(0)).item()
    shift_htc = torch.norm(e_train_htc_base.mean(0) - e_test_htc_base.mean(0)).item()

    print(f"Best-model shift | NB: {shift_nb:.4f} | HTC: {shift_htc:.4f}")

    # =====================================================
    # FEW-SHOT: N-BACK (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    # start NB adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_nb = e_test_nb_base.clone()
    all_y_nb = y_test_nb_base.clone()

    support_idx_nb = []
    for c in torch.unique(all_y_nb):
        class_idx = (all_y_nb == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_nb.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_nb = torch.cat(support_idx_nb)

    mask_nb = torch.zeros(len(all_y_nb), dtype=torch.bool)
    mask_nb[support_idx_nb] = True

    support_subset_nb = torch.utils.data.Subset(test_ds_nb, support_idx_nb.tolist())
    support_loader_nb = DataLoader(support_subset_nb, batch_size=256, shuffle=False)

    use_bn_nb = shift_nb > SHIFT_THRESH
    print(f"N-back BN USED: {use_bn_nb}")

    if use_bn_nb:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_nb:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_nb, all_y_nb = extract_embeddings(model, test_loader_nb, device)

    e_support_nb = all_e_nb[mask_nb].to(device)
    y_support_nb = all_y_nb[mask_nb].to(device)
    e_query_nb   = all_e_nb[~mask_nb].to(device)
    y_query_nb   = all_y_nb[~mask_nb].to(device)

    e_support_nb = F.normalize(e_support_nb, dim=1)
    e_query_nb   = F.normalize(e_query_nb, dim=1)

    var_nb = e_support_nb.var(dim=0)
    weights_nb = 1.0 / (var_nb + 1e-6)
    weights_nb = weights_nb / weights_nb.mean()
    e_support_nb = e_support_nb * weights_nb
    e_query_nb   = e_query_nb * weights_nb

    classes_nb = torch.unique(y_support_nb)
    protos_nb = torch.stack([
        e_support_nb[y_support_nb == c].mean(dim=0) for c in classes_nb
    ])
    protos_nb = F.normalize(protos_nb, dim=1)

    residuals_nb = torch.cat([
        e_support_nb[y_support_nb == c] - protos_nb[i]
        for i, c in enumerate(classes_nb)
    ], dim=0)

    D_nb = residuals_nb.shape[1]
    cov_nb = (residuals_nb.t() @ residuals_nb) / max(residuals_nb.shape[0] - 1, 1)

    avg_var_nb = torch.trace(cov_nb) / D_nb
    cov_shrink_nb = (1 - SHRINK) * cov_nb + SHRINK * (avg_var_nb * torch.eye(D_nb, device=device))
    cov_inv_nb = torch.linalg.pinv(cov_shrink_nb)

    diffs_nb = e_query_nb.unsqueeze(1) - protos_nb.unsqueeze(0)
    dists_nb = torch.einsum("nkd,dd,nkd->nk", diffs_nb, cov_inv_nb, diffs_nb)

    pred_nb_fs = dists_nb.argmin(dim=1)
    fewshot_acc_nb = (pred_nb_fs == y_query_nb).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_nb)

    # =====================================================
    # FEW-SHOT: HTC (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(2234 + fold_i)
    np.random.seed(2234 + fold_i)

    # start HTC adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_htc = e_test_htc_base.clone()
    all_y_htc = y_test_htc_base.clone()

    support_idx_htc = []
    for c in torch.unique(all_y_htc):
        class_idx = (all_y_htc == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_htc.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_htc = torch.cat(support_idx_htc)

    mask_htc = torch.zeros(len(all_y_htc), dtype=torch.bool)
    mask_htc[support_idx_htc] = True

    support_subset_htc = torch.utils.data.Subset(test_ds_htc, support_idx_htc.tolist())
    support_loader_htc = DataLoader(support_subset_htc, batch_size=256, shuffle=False)

    use_bn_htc = shift_htc > SHIFT_THRESH
    print(f"HTC BN USED: {use_bn_htc}")

    if use_bn_htc:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_htc:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_htc, all_y_htc = extract_embeddings(model, test_loader_htc, device)

    e_support_htc = all_e_htc[mask_htc].to(device)
    y_support_htc = all_y_htc[mask_htc].to(device)
    e_query_htc   = all_e_htc[~mask_htc].to(device)
    y_query_htc   = all_y_htc[~mask_htc].to(device)

    e_support_htc = F.normalize(e_support_htc, dim=1)
    e_query_htc   = F.normalize(e_query_htc, dim=1)

    var_htc = e_support_htc.var(dim=0)
    weights_htc = 1.0 / (var_htc + 1e-6)
    weights_htc = weights_htc / weights_htc.mean()
    e_support_htc = e_support_htc * weights_htc
    e_query_htc   = e_query_htc * weights_htc

    classes_htc = torch.unique(y_support_htc)
    protos_htc = torch.stack([
        e_support_htc[y_support_htc == c].mean(dim=0) for c in classes_htc
    ])
    protos_htc = F.normalize(protos_htc, dim=1)

    residuals_htc = torch.cat([
        e_support_htc[y_support_htc == c] - protos_htc[i]
        for i, c in enumerate(classes_htc)
    ], dim=0)

    D_htc = residuals_htc.shape[1]
    cov_htc = (residuals_htc.t() @ residuals_htc) / max(residuals_htc.shape[0] - 1, 1)

    avg_var_htc = torch.trace(cov_htc) / D_htc
    cov_shrink_htc = (1 - SHRINK) * cov_htc + SHRINK * (avg_var_htc * torch.eye(D_htc, device=device))
    cov_inv_htc = torch.linalg.pinv(cov_shrink_htc)

    diffs_htc = e_query_htc.unsqueeze(1) - protos_htc.unsqueeze(0)
    dists_htc = torch.einsum("nkd,dd,nkd->nk", diffs_htc, cov_inv_htc, diffs_htc)

    pred_htc_fs = dists_htc.argmin(dim=1)
    fewshot_acc_htc = (pred_htc_fs == y_query_htc).float().mean().item()

    print("HTC gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_htc)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": best_acc_nb,
        "best_zero_shot_htc": best_acc_htc,
        "shift_nb": shift_nb,
        "shift_htc": shift_htc,
        "bn_used_nb": bool(use_bn_nb),
        "bn_used_htc": bool(use_bn_htc),
        "fewshot_nb": fewshot_acc_nb,
        "fewshot_htc": fewshot_acc_htc
    })

print("\nJOINT METRIC TRAIN + GATED ADABN + MAHALANOBIS LOSO SUMMARY")
for r in overall_results_joint:
    print(
        f"Fold {r['fold']:02d} | "
        f"NB test {r['test_subj_nb']} | HTC test {r['test_subj_htc']} | "
        f"NB zero-shot {r['best_zero_shot_nb']:.4f} | "
        f"HTC zero-shot {r['best_zero_shot_htc']:.4f} | "
        f"NB shift {r['shift_nb']:.4f} | HTC shift {r['shift_htc']:.4f} | "
        f"NB BN {r['bn_used_nb']} | HTC BN {r['bn_used_htc']} | "
        f"NB few-shot {r['fewshot_nb']:.4f} | "
        f"HTC few-shot {r['fewshot_htc']:.4f}"
    )

nb_fs_accs  = [r["fewshot_nb"] for r in overall_results_joint]
htc_fs_accs = [r["fewshot_htc"] for r in overall_results_joint]

print("\nNB few-shot mean/std:", float(np.mean(nb_fs_accs)), float(np.std(nb_fs_accs)))
print("HTC few-shot mean/std:", float(np.mean(htc_fs_accs)), float(np.std(htc_fs_accs)))


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Epoch 01 | NB zero-shot 0.6141 | HTC zero-shot 0.5668 | avg 0.5904
Epoch 02 | NB zero-shot 0.6383 | HTC zero-shot 0.6124 | avg 0.6253
Epoch 03 | NB zero-shot 0.6930 | HTC zero-shot 0.5863 | avg 0.6396
Epoch 04 | NB zero-shot 0.6940 | HTC zero-shot 0.6287 | avg 0.6613
Epoch 05 | NB zero-shot 0.7003 | HTC zero-shot 0.6775 | avg 0.6889
Epoch 06 | NB zero-shot 0.6835 | HTC zero-shot 0.6287 | avg 0.6561
Epoch 07 | NB zero-shot 0.7119 | HTC zero-shot 0.5733 | avg 0.6426
Epoch 08 | NB zero-shot 0.7382 | HTC zero-shot 0.6059 | avg 0.6720
Epoch 09 | NB zero-shot 0.7087 | HTC zero-shot 0.6026 | avg 0.6557
Epoch 10 | NB zero-shot 0.7035 | HTC zero-shot 0.6156 | avg 0.6596
Best-model shift | NB: 0.9747 | HTC: 2.0469
N-back BN USED: True
N-back gated AdaBN + Mahalanobis few-shot acc: 0.7845118045806885
HTC BN USED: True
HTC gated AdaBN + Mahalanobis few-shot acc: 0.8314606547355652

JOINT FOLD 2/13
N-back test subject:

In [ ]:
# ============================================================
# DATASET LEAKAGE TEST
# Can we predict dataset ID from embeddings?
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def extract_all_embeddings(model, loader, device):
    model.eval()
    all_e = []
    all_db = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            all_e.append(e.cpu())
            all_db.append(db.cpu())

    return torch.cat(all_e), torch.cat(all_db)


# extract embeddings from joint train set
e_all, db_all = extract_all_embeddings(model, train_loader, device)

print("Embedding shape:", e_all.shape)

# simple classifier: can we predict dataset?
clf = nn.Sequential(
    nn.Linear(e_all.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

dataset = TensorDataset(e_all, db_all)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

# train quick probe
for epoch in range(10):
    total, correct = 0, 0

    for e_batch, db_batch in loader:
        opt.zero_grad()
        logits = clf(e_batch)
        loss = criterion(logits, db_batch)
        loss.backward()
        opt.step()

        pred = logits.argmax(dim=1)
        total += db_batch.size(0)
        correct += (pred == db_batch).sum().item()

    acc = correct / total
    print(f"Epoch {epoch+1} | dataset clf acc {acc:.4f}")

Embedding shape: torch.Size([11088, 64])
Epoch 1 | dataset clf acc 0.9273
Epoch 2 | dataset clf acc 0.9569
Epoch 3 | dataset clf acc 0.9732
Epoch 4 | dataset clf acc 0.9810
Epoch 5 | dataset clf acc 0.9836
Epoch 6 | dataset clf acc 0.9855
Epoch 7 | dataset clf acc 0.9866
Epoch 8 | dataset clf acc 0.9872
Epoch 9 | dataset clf acc 0.9884
Epoch 10 | dataset clf acc 0.9888


In [ ]:
print("subjects_htc len:", len(subjects_htc))
print("subjects_htc_prefixed len:", len(subjects_htc_prefixed))

subjects_htc len: 5056
subjects_htc_prefixed len: 3880


In [ ]:
# ============================================================
# FIX SUBJECT PREFIX ALIGNMENT
# ============================================================

subjects_htc_prefixed = np.array([
    f"htc_{s}" for s in subjects_htc
])

subjects_nb_prefixed = np.array([
    f"nb_{s}" for s in subjects
])

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))
overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLIT
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb  = np.where(train_mask_nb)[0]
    test_idx_nb   = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc  = np.where(train_mask_htc)[0]
    test_idx_htc   = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0, 2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    # X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    # X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()
    # ============================================================
    # ZERO COV FOR BOTH DATASETS (ANTI-LEAKAGE FIX)
    # ============================================================
    X_cov_train_htc = torch.zeros_like(X_cov_t_htc[train_idx_htc])
    X_cov_test_htc  = torch.zeros_like(X_cov_t_htc[test_idx_htc])

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0, 2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    sampler = JointBalancedBatchSampler(
        train_ds_joint,
        batch_size=BATCH,
        subjects_per_dataset=6
    )

    train_loader         = DataLoader(train_ds_joint, batch_sampler=sampler)
    train_eval_loader_nb = DataLoader(train_ds_nb, batch_size=256, shuffle=False)
    train_eval_loader_htc = DataLoader(train_ds_htc, batch_size=256, shuffle=False)
    test_loader_nb       = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc      = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_score = -1.0
    best_state = None
    best_acc_nb = -1.0
    best_acc_htc = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, loss_metric, loss_con, loss_proto = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL: NB
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(model, train_eval_loader_nb, device)
            e_test_nb,  y_test_nb_full  = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0) for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )

            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            # =========================
            # ZERO-SHOT EVAL: HTC
            # =========================
            e_train_htc, y_train_htc_full = extract_embeddings(model, train_eval_loader_htc, device)
            e_test_htc,  y_test_htc_full  = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0) for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )

            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_acc_nb = acc_nb
            best_acc_htc = acc_htc

        print(f"Epoch {ep:02d} | NB zero-shot {acc_nb:.4f} | HTC zero-shot {acc_htc:.4f} | avg {score:.4f}")

    # =====================================================
    # LOAD BEST BASE MODEL
    # =====================================================
    model.load_state_dict(best_state)
    best_joint_state = copy.deepcopy(best_state)

    # =====================================================
    # SHIFT COMPUTATION ON BEST BASE MODEL
    # =====================================================
    model.eval()
    with torch.no_grad():
        e_train_nb_base, y_train_nb_base = extract_embeddings(model, train_eval_loader_nb, device)
        e_test_nb_base,  y_test_nb_base  = extract_embeddings(model, test_loader_nb, device)

        e_train_htc_base, y_train_htc_base = extract_embeddings(model, train_eval_loader_htc, device)
        e_test_htc_base,  y_test_htc_base  = extract_embeddings(model, test_loader_htc, device)

    shift_nb = torch.norm(e_train_nb_base.mean(0) - e_test_nb_base.mean(0)).item()
    shift_htc = torch.norm(e_train_htc_base.mean(0) - e_test_htc_base.mean(0)).item()

    print(f"Best-model shift | NB: {shift_nb:.4f} | HTC: {shift_htc:.4f}")

    # =====================================================
    # FEW-SHOT: N-BACK (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    # start NB adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_nb = e_test_nb_base.clone()
    all_y_nb = y_test_nb_base.clone()

    support_idx_nb = []
    for c in torch.unique(all_y_nb):
        class_idx = (all_y_nb == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_nb.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_nb = torch.cat(support_idx_nb)

    mask_nb = torch.zeros(len(all_y_nb), dtype=torch.bool)
    mask_nb[support_idx_nb] = True

    support_subset_nb = torch.utils.data.Subset(test_ds_nb, support_idx_nb.tolist())
    support_loader_nb = DataLoader(support_subset_nb, batch_size=256, shuffle=False)

    use_bn_nb = shift_nb > SHIFT_THRESH
    print(f"N-back BN USED: {use_bn_nb}")

    if use_bn_nb:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_nb:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_nb, all_y_nb = extract_embeddings(model, test_loader_nb, device)

    e_support_nb = all_e_nb[mask_nb].to(device)
    y_support_nb = all_y_nb[mask_nb].to(device)
    e_query_nb   = all_e_nb[~mask_nb].to(device)
    y_query_nb   = all_y_nb[~mask_nb].to(device)

    e_support_nb = F.normalize(e_support_nb, dim=1)
    e_query_nb   = F.normalize(e_query_nb, dim=1)

    var_nb = e_support_nb.var(dim=0)
    weights_nb = 1.0 / (var_nb + 1e-6)
    weights_nb = weights_nb / weights_nb.mean()
    e_support_nb = e_support_nb * weights_nb
    e_query_nb   = e_query_nb * weights_nb

    classes_nb = torch.unique(y_support_nb)
    protos_nb = torch.stack([
        e_support_nb[y_support_nb == c].mean(dim=0) for c in classes_nb
    ])
    protos_nb = F.normalize(protos_nb, dim=1)

    residuals_nb = torch.cat([
        e_support_nb[y_support_nb == c] - protos_nb[i]
        for i, c in enumerate(classes_nb)
    ], dim=0)

    D_nb = residuals_nb.shape[1]
    cov_nb = (residuals_nb.t() @ residuals_nb) / max(residuals_nb.shape[0] - 1, 1)

    avg_var_nb = torch.trace(cov_nb) / D_nb
    cov_shrink_nb = (1 - SHRINK) * cov_nb + SHRINK * (avg_var_nb * torch.eye(D_nb, device=device))
    cov_inv_nb = torch.linalg.pinv(cov_shrink_nb)

    diffs_nb = e_query_nb.unsqueeze(1) - protos_nb.unsqueeze(0)
    dists_nb = torch.einsum("nkd,dd,nkd->nk", diffs_nb, cov_inv_nb, diffs_nb)

    pred_nb_fs = dists_nb.argmin(dim=1)
    fewshot_acc_nb = (pred_nb_fs == y_query_nb).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_nb)

    # =====================================================
    # FEW-SHOT: HTC (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(2234 + fold_i)
    np.random.seed(2234 + fold_i)

    # start HTC adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_htc = e_test_htc_base.clone()
    all_y_htc = y_test_htc_base.clone()

    support_idx_htc = []
    for c in torch.unique(all_y_htc):
        class_idx = (all_y_htc == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_htc.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_htc = torch.cat(support_idx_htc)

    mask_htc = torch.zeros(len(all_y_htc), dtype=torch.bool)
    mask_htc[support_idx_htc] = True

    support_subset_htc = torch.utils.data.Subset(test_ds_htc, support_idx_htc.tolist())
    support_loader_htc = DataLoader(support_subset_htc, batch_size=256, shuffle=False)

    use_bn_htc = shift_htc > SHIFT_THRESH
    print(f"HTC BN USED: {use_bn_htc}")

    if use_bn_htc:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_htc:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_htc, all_y_htc = extract_embeddings(model, test_loader_htc, device)

    e_support_htc = all_e_htc[mask_htc].to(device)
    y_support_htc = all_y_htc[mask_htc].to(device)
    e_query_htc   = all_e_htc[~mask_htc].to(device)
    y_query_htc   = all_y_htc[~mask_htc].to(device)

    e_support_htc = F.normalize(e_support_htc, dim=1)
    e_query_htc   = F.normalize(e_query_htc, dim=1)

    var_htc = e_support_htc.var(dim=0)
    weights_htc = 1.0 / (var_htc + 1e-6)
    weights_htc = weights_htc / weights_htc.mean()
    e_support_htc = e_support_htc * weights_htc
    e_query_htc   = e_query_htc * weights_htc

    classes_htc = torch.unique(y_support_htc)
    protos_htc = torch.stack([
        e_support_htc[y_support_htc == c].mean(dim=0) for c in classes_htc
    ])
    protos_htc = F.normalize(protos_htc, dim=1)

    residuals_htc = torch.cat([
        e_support_htc[y_support_htc == c] - protos_htc[i]
        for i, c in enumerate(classes_htc)
    ], dim=0)

    D_htc = residuals_htc.shape[1]
    cov_htc = (residuals_htc.t() @ residuals_htc) / max(residuals_htc.shape[0] - 1, 1)

    avg_var_htc = torch.trace(cov_htc) / D_htc
    cov_shrink_htc = (1 - SHRINK) * cov_htc + SHRINK * (avg_var_htc * torch.eye(D_htc, device=device))
    cov_inv_htc = torch.linalg.pinv(cov_shrink_htc)

    diffs_htc = e_query_htc.unsqueeze(1) - protos_htc.unsqueeze(0)
    dists_htc = torch.einsum("nkd,dd,nkd->nk", diffs_htc, cov_inv_htc, diffs_htc)

    pred_htc_fs = dists_htc.argmin(dim=1)
    fewshot_acc_htc = (pred_htc_fs == y_query_htc).float().mean().item()

    print("HTC gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_htc)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": best_acc_nb,
        "best_zero_shot_htc": best_acc_htc,
        "shift_nb": shift_nb,
        "shift_htc": shift_htc,
        "bn_used_nb": bool(use_bn_nb),
        "bn_used_htc": bool(use_bn_htc),
        "fewshot_nb": fewshot_acc_nb,
        "fewshot_htc": fewshot_acc_htc
    })

print("\nJOINT METRIC TRAIN + GATED ADABN + MAHALANOBIS LOSO SUMMARY")
for r in overall_results_joint:
    print(
        f"Fold {r['fold']:02d} | "
        f"NB test {r['test_subj_nb']} | HTC test {r['test_subj_htc']} | "
        f"NB zero-shot {r['best_zero_shot_nb']:.4f} | "
        f"HTC zero-shot {r['best_zero_shot_htc']:.4f} | "
        f"NB shift {r['shift_nb']:.4f} | HTC shift {r['shift_htc']:.4f} | "
        f"NB BN {r['bn_used_nb']} | HTC BN {r['bn_used_htc']} | "
        f"NB few-shot {r['fewshot_nb']:.4f} | "
        f"HTC few-shot {r['fewshot_htc']:.4f}"
    )

nb_fs_accs  = [r["fewshot_nb"] for r in overall_results_joint]
htc_fs_accs = [r["fewshot_htc"] for r in overall_results_joint]

print("\nNB few-shot mean/std:", float(np.mean(nb_fs_accs)), float(np.std(nb_fs_accs)))
print("HTC few-shot mean/std:", float(np.mean(htc_fs_accs)), float(np.std(htc_fs_accs)))


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Epoch 01 | NB zero-shot 0.6278 | HTC zero-shot 0.6319 | avg 0.6298
Epoch 02 | NB zero-shot 0.6667 | HTC zero-shot 0.6417 | avg 0.6542
Epoch 03 | NB zero-shot 0.6898 | HTC zero-shot 0.5537 | avg 0.6218
Epoch 04 | NB zero-shot 0.7277 | HTC zero-shot 0.5244 | avg 0.6260
Epoch 05 | NB zero-shot 0.7003 | HTC zero-shot 0.5896 | avg 0.6449
Epoch 06 | NB zero-shot 0.6761 | HTC zero-shot 0.6059 | avg 0.6410
Epoch 07 | NB zero-shot 0.6919 | HTC zero-shot 0.5212 | avg 0.6065
Epoch 08 | NB zero-shot 0.6803 | HTC zero-shot 0.5798 | avg 0.6301
Epoch 09 | NB zero-shot 0.6824 | HTC zero-shot 0.6059 | avg 0.6442
Epoch 10 | NB zero-shot 0.6772 | HTC zero-shot 0.5440 | avg 0.6106
Best-model shift | NB: 0.9128 | HTC: 6.0036
N-back BN USED: True
N-back gated AdaBN + Mahalanobis few-shot acc: 0.7620651125907898
HTC BN USED: True
HTC gated AdaBN + Mahalanobis few-shot acc: 0.8614232540130615

JOINT FOLD 2/13
N-back test subject:

In [ ]:
# ============================================================
# DATASET LEAKAGE TEST
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def extract_all_embeddings(model, loader, device):
    model.eval()
    all_e = []
    all_db = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            all_e.append(e.cpu())
            all_db.append(db.cpu())

    return torch.cat(all_e), torch.cat(all_db)


# ============================================================
# EXTRACT EMBEDDINGS FROM JOINT TRAIN SET
# ============================================================

e_all, db_all = extract_all_embeddings(model, train_loader, device)

print("Embedding shape:", e_all.shape)
print("Dataset labels:", torch.unique(db_all))


# ============================================================
# SIMPLE CLASSIFIER TO PREDICT DATASET
# ============================================================

clf = nn.Sequential(
    nn.Linear(e_all.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

clf = clf.to(device)

opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

dataset = TensorDataset(e_all, db_all)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

# ============================================================
# TRAIN PROBE
# ============================================================

for epoch in range(10):
    total, correct = 0, 0

    for e_batch, db_batch in loader:
        e_batch = e_batch.to(device)
        db_batch = db_batch.to(device)

        opt.zero_grad()
        logits = clf(e_batch)
        loss = criterion(logits, db_batch)
        loss.backward()
        opt.step()

        pred = logits.argmax(dim=1)
        total += db_batch.size(0)
        correct += (pred == db_batch).sum().item()

    acc = correct / total
    print(f"Epoch {epoch+1} | dataset clf acc {acc:.4f}")

Embedding shape: torch.Size([11088, 64])
Dataset labels: tensor([0, 1])
Epoch 1 | dataset clf acc 0.9426
Epoch 2 | dataset clf acc 0.9966
Epoch 3 | dataset clf acc 0.9979
Epoch 4 | dataset clf acc 0.9980
Epoch 5 | dataset clf acc 0.9980
Epoch 6 | dataset clf acc 0.9984
Epoch 7 | dataset clf acc 0.9985
Epoch 8 | dataset clf acc 0.9987
Epoch 9 | dataset clf acc 0.9987
Epoch 10 | dataset clf acc 0.9988


In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))
overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLIT
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb  = np.where(train_mask_nb)[0]
    test_idx_nb   = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc  = np.where(train_mask_htc)[0]
    test_idx_htc   = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    # ZERO COV FOR BOTH DATASETS
    X_cov_train_htc = torch.zeros_like(X_cov_t_htc[train_idx_htc])
    X_cov_test_htc  = torch.zeros_like(X_cov_t_htc[test_idx_htc])

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # ============================================================
    # GLOBAL NORMALIZATION FIX
    # Use shared train-set stats across BOTH datasets per fold
    # ============================================================

    # -------------------------
    # TIME
    # -------------------------
    X_time_all_train = torch.cat([X_time_train_nb, X_time_train_htc], dim=0)
    time_mu = X_time_all_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_all_train.std(dim=(0, 2), keepdim=True) + 1e-6

    X_time_train_nb  = (X_time_train_nb  - time_mu) / time_sd
    X_time_test_nb   = (X_time_test_nb   - time_mu) / time_sd
    X_time_train_htc = (X_time_train_htc - time_mu) / time_sd
    X_time_test_htc  = (X_time_test_htc  - time_mu) / time_sd

    # -------------------------
    # BP
    # -------------------------
    X_bp_all_train = torch.cat([X_bp_train_nb, X_bp_train_htc], dim=0)
    bp_mu = X_bp_all_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_all_train.std(dim=0, keepdim=True) + 1e-6

    X_bp_train_nb  = (X_bp_train_nb  - bp_mu) / bp_sd
    X_bp_test_nb   = (X_bp_test_nb   - bp_mu) / bp_sd
    X_bp_train_htc = (X_bp_train_htc - bp_mu) / bp_sd
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu) / bp_sd

    # -------------------------
    # COV
    # still zero for both, but keep this block for shape consistency
    # -------------------------
    X_cov_all_train = torch.cat([X_cov_train_nb, X_cov_train_htc], dim=0)
    cov_mu = X_cov_all_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_all_train.std(dim=0, keepdim=True) + 1e-6

    X_cov_train_nb  = (X_cov_train_nb  - cov_mu) / cov_sd
    X_cov_test_nb   = (X_cov_test_nb   - cov_mu) / cov_sd
    X_cov_train_htc = (X_cov_train_htc - cov_mu) / cov_sd
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu) / cov_sd

    # -------------------------
    # PLV
    # -------------------------
    X_plv_all_train = torch.cat([X_plv_train_nb, X_plv_train_htc], dim=0)
    plv_mu = X_plv_all_train.mean(dim=0, keepdim=True)
    plv_sd = X_plv_all_train.std(dim=0, keepdim=True) + 1e-6

    X_plv_train_nb  = (X_plv_train_nb  - plv_mu) / plv_sd
    X_plv_test_nb   = (X_plv_test_nb   - plv_mu) / plv_sd
    X_plv_train_htc = (X_plv_train_htc - plv_mu) / plv_sd
    X_plv_test_htc  = (X_plv_test_htc  - plv_mu) / plv_sd

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    sampler = JointBalancedBatchSampler(
        train_ds_joint,
        batch_size=BATCH,
        subjects_per_dataset=6
    )

    train_loader          = DataLoader(train_ds_joint, batch_sampler=sampler)
    train_eval_loader_nb  = DataLoader(train_ds_nb, batch_size=256, shuffle=False)
    train_eval_loader_htc = DataLoader(train_ds_htc, batch_size=256, shuffle=False)
    test_loader_nb        = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc       = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_score = -1.0
    best_state = None
    best_acc_nb = -1.0
    best_acc_htc = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, loss_metric, loss_con, loss_proto = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL: NB
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(model, train_eval_loader_nb, device)
            e_test_nb,  y_test_nb_full  = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0) for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )

            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            # =========================
            # ZERO-SHOT EVAL: HTC
            # =========================
            e_train_htc, y_train_htc_full = extract_embeddings(model, train_eval_loader_htc, device)
            e_test_htc,  y_test_htc_full  = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0) for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )

            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_acc_nb = acc_nb
            best_acc_htc = acc_htc

        print(f"Epoch {ep:02d} | NB zero-shot {acc_nb:.4f} | HTC zero-shot {acc_htc:.4f} | avg {score:.4f}")

    # =====================================================
    # LOAD BEST BASE MODEL
    # =====================================================
    model.load_state_dict(best_state)
    best_joint_state = copy.deepcopy(best_state)

    # =====================================================
    # SHIFT COMPUTATION ON BEST BASE MODEL
    # =====================================================
    model.eval()
    with torch.no_grad():
        e_train_nb_base, y_train_nb_base = extract_embeddings(model, train_eval_loader_nb, device)
        e_test_nb_base,  y_test_nb_base  = extract_embeddings(model, test_loader_nb, device)

        e_train_htc_base, y_train_htc_base = extract_embeddings(model, train_eval_loader_htc, device)
        e_test_htc_base,  y_test_htc_base  = extract_embeddings(model, test_loader_htc, device)

    shift_nb = torch.norm(e_train_nb_base.mean(0) - e_test_nb_base.mean(0)).item()
    shift_htc = torch.norm(e_train_htc_base.mean(0) - e_test_htc_base.mean(0)).item()

    print(f"Best-model shift | NB: {shift_nb:.4f} | HTC: {shift_htc:.4f}")

    # =====================================================
    # FEW-SHOT: N-BACK (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_nb = e_test_nb_base.clone()
    all_y_nb = y_test_nb_base.clone()

    support_idx_nb = []
    for c in torch.unique(all_y_nb):
        class_idx = (all_y_nb == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_nb.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_nb = torch.cat(support_idx_nb)

    mask_nb = torch.zeros(len(all_y_nb), dtype=torch.bool)
    mask_nb[support_idx_nb] = True

    support_subset_nb = torch.utils.data.Subset(test_ds_nb, support_idx_nb.tolist())
    support_loader_nb = DataLoader(support_subset_nb, batch_size=256, shuffle=False)

    use_bn_nb = shift_nb > SHIFT_THRESH
    print(f"N-back BN USED: {use_bn_nb}")

    if use_bn_nb:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_nb:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        all_e_nb, all_y_nb = extract_embeddings(model, test_loader_nb, device)

    e_support_nb = all_e_nb[mask_nb].to(device)
    y_support_nb = all_y_nb[mask_nb].to(device)
    e_query_nb   = all_e_nb[~mask_nb].to(device)
    y_query_nb   = all_y_nb[~mask_nb].to(device)

    e_support_nb = F.normalize(e_support_nb, dim=1)
    e_query_nb   = F.normalize(e_query_nb, dim=1)

    var_nb = e_support_nb.var(dim=0)
    weights_nb = 1.0 / (var_nb + 1e-6)
    weights_nb = weights_nb / weights_nb.mean()
    e_support_nb = e_support_nb * weights_nb
    e_query_nb   = e_query_nb * weights_nb

    classes_nb = torch.unique(y_support_nb)
    protos_nb = torch.stack([
        e_support_nb[y_support_nb == c].mean(dim=0) for c in classes_nb
    ])
    protos_nb = F.normalize(protos_nb, dim=1)

    residuals_nb = torch.cat([
        e_support_nb[y_support_nb == c] - protos_nb[i]
        for i, c in enumerate(classes_nb)
    ], dim=0)

    D_nb = residuals_nb.shape[1]
    cov_nb = (residuals_nb.t() @ residuals_nb) / max(residuals_nb.shape[0] - 1, 1)

    avg_var_nb = torch.trace(cov_nb) / D_nb
    cov_shrink_nb = (1 - SHRINK) * cov_nb + SHRINK * (avg_var_nb * torch.eye(D_nb, device=device))
    cov_inv_nb = torch.linalg.pinv(cov_shrink_nb)

    diffs_nb = e_query_nb.unsqueeze(1) - protos_nb.unsqueeze(0)
    dists_nb = torch.einsum("nkd,dd,nkd->nk", diffs_nb, cov_inv_nb, diffs_nb)

    pred_nb_fs = dists_nb.argmin(dim=1)
    fewshot_acc_nb = (pred_nb_fs == y_query_nb).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_nb)

    # =====================================================
    # FEW-SHOT: HTC (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(2234 + fold_i)
    np.random.seed(2234 + fold_i)

    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_htc = e_test_htc_base.clone()
    all_y_htc = y_test_htc_base.clone()

    support_idx_htc = []
    for c in torch.unique(all_y_htc):
        class_idx = (all_y_htc == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_htc.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_htc = torch.cat(support_idx_htc)

    mask_htc = torch.zeros(len(all_y_htc), dtype=torch.bool)
    mask_htc[support_idx_htc] = True

    support_subset_htc = torch.utils.data.Subset(test_ds_htc, support_idx_htc.tolist())
    support_loader_htc = DataLoader(support_subset_htc, batch_size=256, shuffle=False)

    use_bn_htc = shift_htc > SHIFT_THRESH
    print(f"HTC BN USED: {use_bn_htc}")

    if use_bn_htc:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_htc:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        all_e_htc, all_y_htc = extract_embeddings(model, test_loader_htc, device)

    e_support_htc = all_e_htc[mask_htc].to(device)
    y_support_htc = all_y_htc[mask_htc].to(device)
    e_query_htc   = all_e_htc[~mask_htc].to(device)
    y_query_htc   = all_y_htc[~mask_htc].to(device)

    e_support_htc = F.normalize(e_support_htc, dim=1)
    e_query_htc   = F.normalize(e_query_htc, dim=1)

    var_htc = e_support_htc.var(dim=0)
    weights_htc = 1.0 / (var_htc + 1e-6)
    weights_htc = weights_htc / weights_htc.mean()
    e_support_htc = e_support_htc * weights_htc
    e_query_htc   = e_query_htc * weights_htc

    classes_htc = torch.unique(y_support_htc)
    protos_htc = torch.stack([
        e_support_htc[y_support_htc == c].mean(dim=0) for c in classes_htc
    ])
    protos_htc = F.normalize(protos_htc, dim=1)

    residuals_htc = torch.cat([
        e_support_htc[y_support_htc == c] - protos_htc[i]
        for i, c in enumerate(classes_htc)
    ], dim=0)

    D_htc = residuals_htc.shape[1]
    cov_htc = (residuals_htc.t() @ residuals_htc) / max(residuals_htc.shape[0] - 1, 1)

    avg_var_htc = torch.trace(cov_htc) / D_htc
    cov_shrink_htc = (1 - SHRINK) * cov_htc + SHRINK * (avg_var_htc * torch.eye(D_htc, device=device))
    cov_inv_htc = torch.linalg.pinv(cov_shrink_htc)

    diffs_htc = e_query_htc.unsqueeze(1) - protos_htc.unsqueeze(0)
    dists_htc = torch.einsum("nkd,dd,nkd->nk", diffs_htc, cov_inv_htc, diffs_htc)

    pred_htc_fs = dists_htc.argmin(dim=1)
    fewshot_acc_htc = (pred_htc_fs == y_query_htc).float().mean().item()

    print("HTC gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_htc)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": best_acc_nb,
        "best_zero_shot_htc": best_acc_htc,
        "shift_nb": shift_nb,
        "shift_htc": shift_htc,
        "bn_used_nb": bool(use_bn_nb),
        "bn_used_htc": bool(use_bn_htc),
        "fewshot_nb": fewshot_acc_nb,
        "fewshot_htc": fewshot_acc_htc
    })

print("\nJOINT METRIC TRAIN + GATED ADABN + MAHALANOBIS LOSO SUMMARY")
for r in overall_results_joint:
    print(
        f"Fold {r['fold']:02d} | "
        f"NB test {r['test_subj_nb']} | HTC test {r['test_subj_htc']} | "
        f"NB zero-shot {r['best_zero_shot_nb']:.4f} | "
        f"HTC zero-shot {r['best_zero_shot_htc']:.4f} | "
        f"NB shift {r['shift_nb']:.4f} | HTC shift {r['shift_htc']:.4f} | "
        f"NB BN {r['bn_used_nb']} | HTC BN {r['bn_used_htc']} | "
        f"NB few-shot {r['fewshot_nb']:.4f} | "
        f"HTC few-shot {r['fewshot_htc']:.4f}"
    )

nb_fs_accs  = [r["fewshot_nb"] for r in overall_results_joint]
htc_fs_accs = [r["fewshot_htc"] for r in overall_results_joint]

print("\nNB few-shot mean/std:", float(np.mean(nb_fs_accs)), float(np.std(nb_fs_accs)))
print("HTC few-shot mean/std:", float(np.mean(htc_fs_accs)), float(np.std(htc_fs_accs)))


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Epoch 01 | NB zero-shot 0.6225 | HTC zero-shot 0.6840 | avg 0.6533
Epoch 02 | NB zero-shot 0.6751 | HTC zero-shot 0.5603 | avg 0.6177
Epoch 03 | NB zero-shot 0.7361 | HTC zero-shot 0.5668 | avg 0.6514
Epoch 04 | NB zero-shot 0.7119 | HTC zero-shot 0.5733 | avg 0.6426
Epoch 05 | NB zero-shot 0.7182 | HTC zero-shot 0.6026 | avg 0.6604
Epoch 06 | NB zero-shot 0.7634 | HTC zero-shot 0.5798 | avg 0.6716
Epoch 07 | NB zero-shot 0.7497 | HTC zero-shot 0.6254 | avg 0.6876
Epoch 08 | NB zero-shot 0.7392 | HTC zero-shot 0.5798 | avg 0.6595
Epoch 09 | NB zero-shot 0.7844 | HTC zero-shot 0.6221 | avg 0.7033
Epoch 10 | NB zero-shot 0.7203 | HTC zero-shot 0.5700 | avg 0.6452
Best-model shift | NB: 0.9511 | HTC: 2.5950
N-back BN USED: True
N-back gated AdaBN + Mahalanobis few-shot acc: 0.8260381817817688
HTC BN USED: True
HTC gated AdaBN + Mahalanobis few-shot acc: 0.704119861125946

JOINT FOLD 2/13
N-back test subject: 

In [ ]:
# ============================================================
# DATASET LEAKAGE PROBE (FINAL)
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def extract_all_embeddings(model, loader, device):
    model.eval()
    all_e = []
    all_db = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            all_e.append(e.cpu())
            all_db.append(db.cpu())

    return torch.cat(all_e), torch.cat(all_db)


# ============================================================
# STEP 1: EXTRACT EMBEDDINGS
# ============================================================

e_all, db_all = extract_all_embeddings(model, train_loader, device)

print("Embedding shape:", e_all.shape)
print("Dataset labels:", torch.unique(db_all))


# ============================================================
# STEP 2: TRAIN DATASET CLASSIFIER
# ============================================================

clf = nn.Sequential(
    nn.Linear(e_all.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, 2)
).to(device)

opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

dataset = TensorDataset(e_all, db_all)
loader = DataLoader(dataset, batch_size=256, shuffle=True)


# ============================================================
# STEP 3: TRAIN PROBE
# ============================================================

for epoch in range(10):
    total, correct = 0, 0

    for e_batch, db_batch in loader:
        e_batch = e_batch.to(device)
        db_batch = db_batch.to(device)

        opt.zero_grad()
        logits = clf(e_batch)
        loss = criterion(logits, db_batch)
        loss.backward()
        opt.step()

        pred = logits.argmax(dim=1)
        total += db_batch.size(0)
        correct += (pred == db_batch).sum().item()

    acc = correct / total
    print(f"Epoch {epoch+1} | dataset clf acc {acc:.4f}")

Embedding shape: torch.Size([11088, 64])
Dataset labels: tensor([0, 1])
Epoch 1 | dataset clf acc 0.9223
Epoch 2 | dataset clf acc 0.9972
Epoch 3 | dataset clf acc 0.9980
Epoch 4 | dataset clf acc 0.9982
Epoch 5 | dataset clf acc 0.9985
Epoch 6 | dataset clf acc 0.9986
Epoch 7 | dataset clf acc 0.9987
Epoch 8 | dataset clf acc 0.9986
Epoch 9 | dataset clf acc 0.9989
Epoch 10 | dataset clf acc 0.9989
